In [1]:
import sys

print(sys.executable)

D:\BaiduNetdiskDownload\Anaconda3\envs\geosimcp_repro\python.exe


# 西雅图住房价格统一三段划分基线实验

## 实验目的

本实验在统一的数据划分和评价协议下，依次比较以下方法：

1. 传统 Conformal Prediction（CP）；
2. Geographical Conformal Prediction（GeoCP）；
3. GeoSIMCP-EUC；
4. GeoSIMCP-MND。

完成上述基线实验后，将在相同数据、基础预测模型、划分方式和评价指标下，引入GCCM因果信息，构建并评价GCCP。

## 外层数据划分

全部3000条西雅图住房样本划分为：

- **Train：70%**，用于训练基础预测模型和在训练集内部选择超参数；
- **Calibration：15%**，用于计算最终非一致性分数和共形分位数；
- **Test：15%**，只用于所有参数确定后的最终评价。

Calibration和Test均不参与GeoCP、GeoSIMCP及后续GCCP的超参数选择。

## 公平比较原则

所有方法统一使用：

- 相同的数据划分索引；
- 相同的XGBoost基础预测模型；
- 相同的住房价格预测目标；
- 相同的非一致性分数；
- 相同的目标覆盖率；
- 相同的校准集和测试集；
- 相同的评价指标。

CP、GeoCP和GeoSIMCP尽量保持参考文献及作者公开实现中的方法定义。GCCP在此基础上引入GCCM因果信息，以检验因果相似性是否能够提供普通属性相似性之外的预测区间改进。

## 1. 实验环境与项目路径配置

本节导入实验所需的Python库，并配置西雅图住房数据、作者源代码以及实验输出文件夹的路径。

由于本实验需要依次实现CP、GeoCP、GeoSIMCP-EUC和GeoSIMCP-MND，因此首先检查作者代码目录中实际存在的Python文件及其类名，避免因代码版本不同而调用错误。

In [2]:
# ============================================================
# 第1章：实验环境与项目路径配置
# ============================================================

from pathlib import Path
import sys
import ast
import platform

import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
import xgboost as xgb

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)


# ------------------------------------------------------------
# 1. 固定实验设置
# ------------------------------------------------------------

RANDOM_STATE = 42
ALPHA = 0.1
TARGET_COVERAGE = 1 - ALPHA

np.random.seed(RANDOM_STATE)


# ------------------------------------------------------------
# 2. 项目路径
# ------------------------------------------------------------

PROJECT_ROOT = Path(
    r"E:\GeoSIMCP_Reproduce\geoconformal-main"
)

DATA_PATH = (
    PROJECT_ROOT
    / "example"
    / "seattle_sample_3k.csv"
)

MANUSCRIPT_CODE_DIR = (
    PROJECT_ROOT
    / "Codes_Manuscript_GeoSIMCP"
    / "Codes_GeoSIMCP"
    / "GeoConformal"
)

EXPERIMENT_ROOT = (
    PROJECT_ROOT
    / "GCCP_Seattle"
)

OUTPUT_DIR = (
    EXPERIMENT_ROOT
    / "outputs"
)


# ------------------------------------------------------------
# 3. 创建输出文件夹
# ------------------------------------------------------------

OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

SPLIT_PATH = (
    OUTPUT_DIR
    / "seattle_split_seed42.npz"
)

BASELINE_RESULT_PATH = (
    OUTPUT_DIR
    / "baseline_results_seed42.csv"
)


# ------------------------------------------------------------
# 4. 检查关键路径
# ------------------------------------------------------------

if not PROJECT_ROOT.exists():
    raise FileNotFoundError(
        f"项目根目录不存在：{PROJECT_ROOT}"
    )

if not DATA_PATH.exists():
    raise FileNotFoundError(
        f"住房数据不存在：{DATA_PATH}"
    )

if not MANUSCRIPT_CODE_DIR.exists():
    raise FileNotFoundError(
        f"作者源代码目录不存在：{MANUSCRIPT_CODE_DIR}"
    )


# ------------------------------------------------------------
# 5. 将作者代码目录加入Python搜索路径
# ------------------------------------------------------------

code_directory_string = str(
    MANUSCRIPT_CODE_DIR
)

if code_directory_string not in sys.path:
    sys.path.insert(
        0,
        code_directory_string
    )


# ------------------------------------------------------------
# 6. 查看作者代码目录中的Python文件和类
# ------------------------------------------------------------

source_records = []

for python_file in sorted(
    MANUSCRIPT_CODE_DIR.glob("*.py")
):
    try:
        source_text = python_file.read_text(
            encoding="utf-8"
        )

        syntax_tree = ast.parse(
            source_text
        )

        class_names = [
            node.name
            for node in syntax_tree.body
            if isinstance(node, ast.ClassDef)
        ]

        function_names = [
            node.name
            for node in syntax_tree.body
            if isinstance(node, ast.FunctionDef)
        ]

        source_records.append({
            "Python file": python_file.name,
            "Classes": ", ".join(class_names),
            "Functions": ", ".join(function_names)
        })

    except Exception as error:
        source_records.append({
            "Python file": python_file.name,
            "Classes": f"读取失败：{error}",
            "Functions": ""
        })


source_inventory = pd.DataFrame(
    source_records
)

display(source_inventory)


# ------------------------------------------------------------
# 7. 输出实验环境信息
# ------------------------------------------------------------

print("第1章检查完成。")
print("Python版本：", platform.python_version())
print("NumPy版本：", np.__version__)
print("Pandas版本：", pd.__version__)
print("GeoPandas版本：", gpd.__version__)
print("XGBoost版本：", xgb.__version__)
print()
print("项目根目录：", PROJECT_ROOT)
print("住房数据：", DATA_PATH)
print("作者代码目录：", MANUSCRIPT_CODE_DIR)
print("实验输出目录：", OUTPUT_DIR)
print("数据划分保存位置：", SPLIT_PATH)
print("基线结果保存位置：", BASELINE_RESULT_PATH)

,Python file,Classes,Functions
0,__init__.py,,
1,GeoConformalSpatialRegression.py,GeoConformalSpatialRegression,
2,GeoSIMCP.py,GeoSIMConformalSpatialRegression,
3,GeoSIMCP_non_zero.py,GeoSIMConformalSpatialRegression,
4,GeoSIMCP_test.py,GeoSIMConformalSpatialRegression,
5,GeoSIMCPzhao.py,GeoSIMCPzhao,
6,GWQRBasedGeoCP.py,GWQRBasedGeoConformalSpatialRegression,
7,LSCP.py,LSCP,
8,utils.py,GeoConformalResults,


第1章检查完成。
Python版本： 3.10.20
NumPy版本： 2.2.6
Pandas版本： 2.3.3
GeoPandas版本： 1.1.4
XGBoost版本： 3.2.0

项目根目录： E:\GeoSIMCP_Reproduce\geoconformal-main
住房数据： E:\GeoSIMCP_Reproduce\geoconformal-main\example\seattle_sample_3k.csv
作者代码目录： E:\GeoSIMCP_Reproduce\geoconformal-main\Codes_Manuscript_GeoSIMCP\Codes_GeoSIMCP\GeoConformal
实验输出目录： E:\GeoSIMCP_Reproduce\geoconformal-main\GCCP_Seattle\outputs
数据划分保存位置： E:\GeoSIMCP_Reproduce\geoconformal-main\GCCP_Seattle\outputs\seattle_split_seed42.npz
基线结果保存位置： E:\GeoSIMCP_Reproduce\geoconformal-main\GCCP_Seattle\outputs\baseline_results_seed42.csv


### 1.1 作者代码接口检查

本地作者代码目录中包含GeoCP及多个GeoSIMCP实现版本。由于不同文件中的类名相同，但距离定义和参数接口可能不同，本节直接读取源代码，检查：

- 类的初始化参数；
- 可调用的主要方法；
- 是否包含欧氏距离、MND距离、带宽和融合权重等关键词。

后续基线实验将依据实际源代码选择实现，而不是仅根据文件名判断。

In [3]:
# ============================================================
# 检查GeoCP和不同GeoSIMCP版本的接口及距离定义
# ============================================================

FILES_TO_INSPECT = [
    "GeoConformalSpatialRegression.py",
    "GeoSIMCP.py",
    "GeoSIMCP_non_zero.py",
    "GeoSIMCP_test.py",
    "GeoSIMCPzhao.py"
]

interface_records = []
keyword_records = []

keywords = [
    "euclidean",
    "mnd",
    "distance_metric",
    "distance",
    "bandwidth",
    "lambda",
    "weight",
    "quantile"
]

for file_name in FILES_TO_INSPECT:
    file_path = MANUSCRIPT_CODE_DIR / file_name
    source_text = file_path.read_text(encoding="utf-8")
    syntax_tree = ast.parse(source_text)

    # --------------------------------------------------------
    # 1. 读取类名、__init__参数和公开方法
    # --------------------------------------------------------
    for node in syntax_tree.body:
        if isinstance(node, ast.ClassDef):

            init_parameters = []
            public_methods = []

            for class_item in node.body:
                if isinstance(class_item, ast.FunctionDef):

                    if class_item.name == "__init__":
                        init_parameters = [
                            argument.arg
                            for argument in class_item.args.args
                            if argument.arg != "self"
                        ]

                    elif not class_item.name.startswith("_"):
                        public_methods.append(class_item.name)

            interface_records.append({
                "Python file": file_name,
                "Class": node.name,
                "__init__ parameters": ", ".join(init_parameters),
                "Public methods": ", ".join(public_methods)
            })

    # --------------------------------------------------------
    # 2. 提取包含关键距离词的源代码行
    # --------------------------------------------------------
    for line_number, line_text in enumerate(
        source_text.splitlines(),
        start=1
    ):
        stripped_line = line_text.strip()
        lower_line = stripped_line.lower()

        matched_keywords = [
            keyword
            for keyword in keywords
            if keyword in lower_line
        ]

        if matched_keywords and stripped_line:
            keyword_records.append({
                "Python file": file_name,
                "Line": line_number,
                "Keyword": ", ".join(matched_keywords),
                "Source code": stripped_line
            })


interface_table = pd.DataFrame(interface_records)
keyword_table = pd.DataFrame(keyword_records)

print("各类接口：")
display(interface_table)

print("\n与距离、带宽和权重相关的源代码：")
display(keyword_table)

各类接口：


,Python file,Class,__init__ parameters,Public methods
0,GeoConformalSpatialRegression.py,GeoConformalSpatialRegression,"predict_f, nonconformity_score_f, miscoverage_...","predict_geoconformal_uncertainty, predict_conf..."
1,GeoSIMCP.py,GeoSIMConformalSpatialRegression,"predict_f, nonconformity_score_f, miscoverage_...","predict_geoconformal_uncertainty, predict_conf..."
2,GeoSIMCP_non_zero.py,GeoSIMConformalSpatialRegression,"predict_f, nonconformity_score_f, miscoverage_...","predict_geoconformal_uncertainty, predict_conf..."
3,GeoSIMCP_test.py,GeoSIMConformalSpatialRegression,"predict_f, nonconformity_score_f, miscoverage_...","predict_geoconformal_uncertainty, predict_conf..."
4,GeoSIMCPzhao.py,GeoSIMCPzhao,"predict_f, nonconformity_score_f, miscoverage_...","predict_geoconformal_uncertainty, predict_conf..."



与距离、带宽和权重相关的源代码：


,Python file,Line,Keyword,Source code
0,GeoConformalSpatialRegression.py,10,bandwidth,"miscoverage_level: float = 0.1, bandwidth: flo..."
1,GeoConformalSpatialRegression.py,20,bandwidth,bandwidth: the bandwidth of the Gaussian kernel
2,GeoConformalSpatialRegression.py,28,bandwidth,self.bandwidth = bandwidth
3,GeoConformalSpatialRegression.py,45,weight,Calculate the geographically weighted uncertai...
4,GeoConformalSpatialRegression.py,53,"bandwidth, weight","weights = self._kernel_smoothing(p, self.coord..."
...,...,...,...,...
143,GeoSIMCPzhao.py,122,weight,weights_sorted = weights[sorted_indices]
144,GeoSIMCPzhao.py,124,weight,cumsum_weights = np.cumsum(weights_sorted)
145,GeoSIMCPzhao.py,125,weight,if cumsum_weights[-1] == 0:
146,GeoSIMCPzhao.py,128,weight,normalized_cumsum_weights = cumsum_weights / c...


### 1.2 距离公式与类接口的精确识别

上一节确认了作者代码中存在多个GeoSIMCP版本，但表格显示内容被截断。

本节进一步输出各类的完整初始化参数，并提取地理距离、特征距离、融合权重、核权重和加权分位数相关的关键代码，从而确定各文件对应的具体方法。

In [4]:
# ============================================================
# 精确打印GeoCP和GeoSIMCP各版本的接口与关键距离公式
# ============================================================

FILES_TO_INSPECT = [
    "GeoConformalSpatialRegression.py",
    "GeoSIMCP.py",
    "GeoSIMCP_non_zero.py",
    "GeoSIMCP_test.py",
    "GeoSIMCPzhao.py"
]

important_patterns = [
    "def __init__",
    "np.linalg.norm",
    "cdist",
    "euclidean",
    "mnd",
    "distance_metric",
    "d_geo",
    "d_feat",
    "joint_distance",
    "combined_distance",
    "lambda_weight",
    "self.lambda",
    "bandwidth",
    "kernel_smoothing",
    "weights =",
    "weighted_quantile",
    "quantile"
]

for file_name in FILES_TO_INSPECT:

    file_path = MANUSCRIPT_CODE_DIR / file_name
    source_text = file_path.read_text(encoding="utf-8")
    source_lines = source_text.splitlines()
    syntax_tree = ast.parse(source_text)

    print("\n" + "=" * 90)
    print("文件：", file_name)
    print("=" * 90)

    # --------------------------------------------------------
    # 1. 完整打印类名和__init__参数
    # --------------------------------------------------------
    for node in syntax_tree.body:
        if isinstance(node, ast.ClassDef):

            print("类名：", node.name)

            for class_item in node.body:
                if (
                    isinstance(class_item, ast.FunctionDef)
                    and class_item.name == "__init__"
                ):
                    parameters = [
                        argument.arg
                        for argument in class_item.args.args
                        if argument.arg != "self"
                    ]

                    print("__init__参数：")
                    for parameter in parameters:
                        print("  -", parameter)

    # --------------------------------------------------------
    # 2. 打印与距离、权重和分位数有关的关键源代码
    # --------------------------------------------------------
    print("\n关键源代码：")

    matched_count = 0

    for line_number, line_text in enumerate(
        source_lines,
        start=1
    ):
        stripped_line = line_text.strip()
        lower_line = stripped_line.lower()

        if any(
            pattern.lower() in lower_line
            for pattern in important_patterns
        ):
            print(
                f"{line_number:>4}: {stripped_line}"
            )
            matched_count += 1

    if matched_count == 0:
        print("未找到关键代码行。")


文件： GeoConformalSpatialRegression.py
类名： GeoConformalSpatialRegression
__init__参数：
  - predict_f
  - nonconformity_score_f
  - miscoverage_level
  - bandwidth
  - coord_calib
  - coord_test
  - X_calib
  - y_calib
  - X_test
  - y_test

关键源代码：
   9: def __init__(self, predict_f: Callable, nonconformity_score_f: Callable = None,
  10: miscoverage_level: float = 0.1, bandwidth: float = None,
  20: bandwidth: the bandwidth of the Gaussian kernel
  28: self.bandwidth = bandwidth
  53: weights = self._kernel_smoothing(p, self.coord_calib, self.bandwidth)
  54: quantile = self._weighted_quantile(nonconformity_scores, self.miscoverage_level, weights)
  55: uncertainty_list.append(quantile)
  56: uncertainty = np.quantile(nonconformity_scores, 1 - self.miscoverage_level)
 102: def _kernel_smoothing(self, z_test: np.ndarray, z_calib: np.ndarray, bandwidth: float) -> np.ndarray:
 107: :param bandwidth: distance decay parameter
 112: weights = self._gaussian_kernel(distances / bandwidth)
 115: d

In [5]:
# ============================================================
# 查看 GeoSIMCPzhao.py 中自定义特征距离的完整公式
# ============================================================

file_path = (
    MANUSCRIPT_CODE_DIR
    / "GeoSIMCPzhao.py"
)

source_text = file_path.read_text(
    encoding="utf-8"
)

source_lines = source_text.splitlines()
syntax_tree = ast.parse(source_text)

method_found = False

for node in syntax_tree.body:
    if isinstance(node, ast.ClassDef):

        for class_item in node.body:
            if (
                isinstance(class_item, ast.FunctionDef)
                and class_item.name == "_custom_feat_distance"
            ):
                method_found = True

                print("=" * 80)
                print("文件：GeoSIMCPzhao.py")
                print("类名：", node.name)
                print("方法：_custom_feat_distance")
                print("=" * 80)

                for line_number in range(
                    class_item.lineno,
                    class_item.end_lineno + 1
                ):
                    print(
                        f"{line_number:>4}: "
                        f"{source_lines[line_number - 1]}"
                    )

if not method_found:
    print("没有找到 _custom_feat_distance 方法。")

文件：GeoSIMCPzhao.py
类名： GeoSIMCPzhao
方法：_custom_feat_distance
  56:     def _custom_feat_distance(self, x_test, x_calib):
  57:         diffs = np.abs(x_calib - x_test)
  58:         scaled = 1 - (diffs / self.ranges)
  59:         similarity = np.min(scaled, axis=1)
  60:         return 1 - similarity


In [6]:
# ============================================================
# 查看 GeoSIMCPzhao.py 中与特征范围和标准化有关的代码
# ============================================================

search_terms = [
    "standardize",
    "x_min",
    "x_max",
    "min(",
    "max(",
    "range",
    "ptp",
    "x_calib_weight",
    "x_test_weight"
]

for line_number, line_text in enumerate(
    source_lines,
    start=1
):
    lower_line = line_text.lower()

    if any(
        term.lower() in lower_line
        for term in search_terms
    ):
        print(
            f"{line_number:>4}: {line_text}"
        )

  15:                  X_calib_weight=None, X_test_weight=None,
  16:                  standardize_weights: bool = True):
  33:         self.X_calib_weight = self._to_numpy(X_calib_weight) if X_calib_weight is not None else self.X_calib
  34:         self.X_test_weight = self._to_numpy(X_test_weight) if X_test_weight is not None else self.X_test
  36:         if standardize_weights:
  37:             self.ranges = np.ptp(self.X_calib_weight, axis=0)  # feature-wise range
  38:             self.ranges[self.ranges == 0] = 1e-8  # avoid zero range
  40:             self.ranges = np.ones(self.X_calib_weight.shape[1])
  58:         scaled = 1 - (diffs / self.ranges)
  59:         similarity = np.min(scaled, axis=1)
  76:         for i in range(len(self.coord_test)):
  78:             x_test_i = self.X_test_weight[i]
  83:                 x_calib=self.X_calib_weight,
 130:         return scores_sorted[min(idx, len(scores_sorted) - 1)]


### 1.5 其他GeoSIMCP代码版本说明

作者代码目录还包含 `GeoSIMCP_non_zero.py` 和
`GeoSIMCP_test.py`。源代码检查表明，这两个文件均使用欧氏特征
距离，因此不属于独立的GeoSIMCP方法。

`GeoSIMCP_non_zero.py` 在距离、带宽和权重归一化过程中加入
`1e-8`，主要用于避免除零及数值不稳定问题，可以视为
GeoSIMCP-EUC的数值稳定版本。

`GeoSIMCP_test.py` 与GeoSIMCP-EUC使用相同的联合距离公式，
但接口更为简化，并额外保存全局共形分位数，属于测试或中间
实现版本。

因此，本实验不将上述两个文件作为独立baseline。正式比较采用：

- `GeoSIMCP.py`：GeoSIMCP-EUC；
- `GeoSIMCPzhao.py`：GeoSIMCP-MND。

## 2. 数据读取、变量构造与统一三段划分

本节读取西雅图住房价格数据，并按照作者住房实验的变量设置构造预测目标、住房属性特征和空间坐标。

预测目标为：

\[
price = \frac{10^{log\_price}}{10000}
\]

因此，`price` 的单位为万美元。

实验使用以下三类输入：

- **非空间住房属性**：用于GeoSIMCP的特征相似性计算；
- **住房属性与UTM坐标**：用于训练XGBoost点预测模型；
- **经纬度坐标**：用于GeoCP和GeoSIMCP的地理距离计算。

全部3000条样本按照固定随机种子42划分为：

- Train：2100条；
- Calibration：450条；
- Test：450条。

三部分样本索引将保存到文件中。后续CP、GeoCP、GeoSIMCP和GCCP均读取同一份索引，避免不同方法因重新随机划分而产生不公平比较。

In [7]:
# ============================================================
# 第2章第1步：读取住房数据并构造实验变量
# ============================================================

# ------------------------------------------------------------
# 1. 读取原始数据
# ------------------------------------------------------------

data = pd.read_csv(DATA_PATH)

print("原始数据形状：", data.shape)
display(data.head())


# ------------------------------------------------------------
# 2. 检查数据字段
# ------------------------------------------------------------

print("\n数据字段：")
for column_name in data.columns:
    print(" -", column_name)


# ------------------------------------------------------------
# 3. 构造空间坐标
# 原始坐标为UTM Zone 10N，转换为经纬度
# ------------------------------------------------------------

geometry = gpd.points_from_xy(
    data["UTM_X"],
    data["UTM_Y"]
)

housing_gdf = gpd.GeoDataFrame(
    data.copy(),
    geometry=geometry,
    crs="EPSG:32610"
)

housing_gdf = housing_gdf.to_crs(
    "EPSG:4326"
)

data["longitude"] = housing_gdf.geometry.x
data["latitude"] = housing_gdf.geometry.y


# ------------------------------------------------------------
# 4. 构造住房价格目标变量
# 单位：万美元
# ------------------------------------------------------------

data["price"] = (
    np.power(10, data["log_price"])
    / 10000
)


# ------------------------------------------------------------
# 5. 定义特征变量
# ------------------------------------------------------------

ASpatial_FEATURES = [
    "bathrooms",
    "sqft_living",
    "sqft_lot",
    "grade",
    "condition",
    "waterfront",
    "view",
    "age"
]

SPATIAL_FEATURES = (
    ASpatial_FEATURES
    + ["UTM_X", "UTM_Y"]
)


# 非空间住房属性：用于EUC、MND及后续GCCM因果距离
X_aspatial = data[
    ASpatial_FEATURES
].copy()

# 住房属性与空间坐标：用于XGBoost点预测
X_spatial = data[
    SPATIAL_FEATURES
].copy()

# 预测目标
y = data["price"].copy()

# GeoCP和GeoSIMCP使用的地理坐标
loc = data[
    ["longitude", "latitude"]
].copy()


# ------------------------------------------------------------
# 6. 基础数据检查
# ------------------------------------------------------------

missing_summary = pd.DataFrame({
    "Variable": (
        ASpatial_FEATURES
        + ["UTM_X", "UTM_Y", "longitude", "latitude", "price"]
    ),
    "Missing values": [
        int(data[column].isna().sum())
        for column in (
            ASpatial_FEATURES
            + ["UTM_X", "UTM_Y", "longitude", "latitude", "price"]
        )
    ]
})

display(missing_summary)

print("X_aspatial形状：", X_aspatial.shape)
print("X_spatial形状：", X_spatial.shape)
print("y形状：", y.shape)
print("loc形状：", loc.shape)

print("\n房价范围（万美元）：")
print("最小值：", round(float(y.min()), 4))
print("最大值：", round(float(y.max()), 4))
print("平均值：", round(float(y.mean()), 4))
print("中位数：", round(float(y.median()), 4))

print("\n重复记录数量：", int(data.duplicated().sum()))


# ------------------------------------------------------------
# 7. 必要断言
# ------------------------------------------------------------

assert len(data) == 3000
assert X_aspatial.shape == (3000, 8)
assert X_spatial.shape == (3000, 10)
assert loc.shape == (3000, 2)

assert not X_aspatial.isna().any().any()
assert not X_spatial.isna().any().any()
assert not y.isna().any()
assert not loc.isna().any().any()

print("\n第2章第1步完成：住房数据和实验变量构造正确。")

原始数据形状： (3000, 12)


,Unnamed: 0,bathrooms,sqft_living,sqft_lot,grade,condition,waterfront,view,age,UTM_X,UTM_Y,log_price
0,0,3.00,2660,4600,8,3,0,0,109,552217.557035,5.274945e+06,6.091315
1,1,2.25,2530,8736,7,4,0,0,57,565692.484331,5.272758e+06,5.790988
2,2,2.00,1390,13464,7,4,0,0,28,562451.661509,5.245291e+06,5.315130
3,3,1.00,940,4264,7,5,0,0,66,546816.935618,5.264407e+06,5.619093
4,4,2.25,2070,7225,8,3,0,0,36,564343.195352,5.244978e+06,5.477121



数据字段：
 - Unnamed: 0
 - bathrooms
 - sqft_living
 - sqft_lot
 - grade
 - condition
 - waterfront
 - view
 - age
 - UTM_X
 - UTM_Y
 - log_price


,Variable,Missing values
0,bathrooms,0
1,sqft_living,0
2,sqft_lot,0
3,grade,0
4,condition,0
5,waterfront,0
6,view,0
7,age,0
8,UTM_X,0
9,UTM_Y,0


X_aspatial形状： (3000, 8)
X_spatial形状： (3000, 10)
y形状： (3000,)
loc形状： (3000, 2)

房价范围（万美元）：
最小值： 8.2
最大值： 363.5
平均值： 52.8893
中位数： 43.5

重复记录数量： 0

第2章第1步完成：住房数据和实验变量构造正确。


In [8]:
# ============================================================
# 第2章第2步：建立并保存统一三段划分
# ============================================================

all_indices = np.arange(len(data))


# ------------------------------------------------------------
# 1. 若划分文件已经存在，则读取原有索引
# 避免重复运行时改变各方法使用的数据
# ------------------------------------------------------------

if SPLIT_PATH.exists():

    saved_split = np.load(SPLIT_PATH)

    train_indices = saved_split["train_indices"]
    calibration_indices = saved_split["calibration_indices"]
    test_indices = saved_split["test_indices"]

    print("已读取以前保存的三段划分：")
    print(SPLIT_PATH)


# ------------------------------------------------------------
# 2. 若划分文件不存在，则创建并保存
# ------------------------------------------------------------

else:

    # 首先划分70%的Train和30%的剩余样本
    train_indices, remaining_indices = train_test_split(
        all_indices,
        test_size=0.30,
        random_state=RANDOM_STATE,
        shuffle=True
    )

    # 将剩余30%平均划分为Calibration和Test
    calibration_indices, test_indices = train_test_split(
        remaining_indices,
        test_size=0.50,
        random_state=RANDOM_STATE,
        shuffle=True
    )

    np.savez(
        SPLIT_PATH,
        train_indices=train_indices,
        calibration_indices=calibration_indices,
        test_indices=test_indices,
        random_state=np.array([RANDOM_STATE])
    )

    print("已创建并保存新的三段划分：")
    print(SPLIT_PATH)


# ------------------------------------------------------------
# 3. 检查三部分数量和是否重叠
# ------------------------------------------------------------

train_set = set(train_indices.tolist())
calibration_set = set(calibration_indices.tolist())
test_set = set(test_indices.tolist())

assert len(train_indices) == 2100
assert len(calibration_indices) == 450
assert len(test_indices) == 450

assert train_set.isdisjoint(calibration_set)
assert train_set.isdisjoint(test_set)
assert calibration_set.isdisjoint(test_set)

assert len(
    train_set | calibration_set | test_set
) == len(data)


# ------------------------------------------------------------
# 4. 根据固定索引生成三部分数据
# ------------------------------------------------------------

X_aspatial_train = X_aspatial.iloc[
    train_indices
].copy()

X_aspatial_calibration = X_aspatial.iloc[
    calibration_indices
].copy()

X_aspatial_test = X_aspatial.iloc[
    test_indices
].copy()


X_spatial_train = X_spatial.iloc[
    train_indices
].copy()

X_spatial_calibration = X_spatial.iloc[
    calibration_indices
].copy()

X_spatial_test = X_spatial.iloc[
    test_indices
].copy()


y_train = y.iloc[
    train_indices
].copy()

y_calibration = y.iloc[
    calibration_indices
].copy()

y_test = y.iloc[
    test_indices
].copy()


loc_train = loc.iloc[
    train_indices
].copy()

loc_calibration = loc.iloc[
    calibration_indices
].copy()

loc_test = loc.iloc[
    test_indices
].copy()


# ------------------------------------------------------------
# 5. 汇总三段数据
# ------------------------------------------------------------

split_summary = pd.DataFrame({
    "Dataset": [
        "Train",
        "Calibration",
        "Test"
    ],
    "Samples": [
        len(train_indices),
        len(calibration_indices),
        len(test_indices)
    ],
    "Proportion": [
        len(train_indices) / len(data),
        len(calibration_indices) / len(data),
        len(test_indices) / len(data)
    ],
    "Mean price": [
        y_train.mean(),
        y_calibration.mean(),
        y_test.mean()
    ],
    "Median price": [
        y_train.median(),
        y_calibration.median(),
        y_test.median()
    ]
})

display(split_summary.round(4))

print("Train、Calibration、Test互不重叠：是")
print("三部分样本总数：", len(data))
print("统一索引文件：", SPLIT_PATH)
print("\n第2章完成：统一三段划分已经建立并保存。")

已读取以前保存的三段划分：
E:\GeoSIMCP_Reproduce\geoconformal-main\GCCP_Seattle\outputs\seattle_split_seed42.npz


,Dataset,Samples,Proportion,Mean price,Median price
0,Train,2100,0.70,53.1103,43.42
1,Calibration,450,0.15,53.5939,45.00
2,Test,450,0.15,51.1534,43.25


Train、Calibration、Test互不重叠：是
三部分样本总数： 3000
统一索引文件： E:\GeoSIMCP_Reproduce\geoconformal-main\GCCP_Seattle\outputs\seattle_split_seed42.npz

第2章完成：统一三段划分已经建立并保存。


## 3. Train内部超参数选择划分

外层实验采用Train、Calibration和Test三段划分。

为了避免使用外层Calibration或Test选择GeoCP、GeoSIMCP及后续GCCP的超参数，本实验进一步在外层Train内部建立临时调参划分：

- **Inner Fit：70%**，用于训练临时XGBoost模型；
- **Inner Calibration：15%**，用于计算临时校准残差；
- **Inner Validation：15%**，用于选择bandwidth、lambda等方法参数。

外层Calibration仍只用于参数确定后的正式共形校准；外层Test仍只用于最终评价。因此，Train内部划分不会改变外层三段实验设计。

对于外层Train中的2100条样本，内部划分为：

- Inner Fit：1470条；
- Inner Calibration：315条；
- Inner Validation：315条。

In [9]:
# ============================================================
# 第3章：在外层Train内部建立临时调参划分
# ============================================================

INNER_SPLIT_PATH = (
    OUTPUT_DIR
    / "seattle_inner_split_seed42.npz"
)


# ------------------------------------------------------------
# 1. 若内部划分已经保存，则直接读取
# ------------------------------------------------------------

if INNER_SPLIT_PATH.exists():

    saved_inner_split = np.load(
        INNER_SPLIT_PATH
    )

    inner_fit_indices = saved_inner_split[
        "inner_fit_indices"
    ]

    inner_calibration_indices = saved_inner_split[
        "inner_calibration_indices"
    ]

    inner_validation_indices = saved_inner_split[
        "inner_validation_indices"
    ]

    print("已读取以前保存的Train内部划分：")
    print(INNER_SPLIT_PATH)


# ------------------------------------------------------------
# 2. 若不存在，则仅在外层Train中创建
# ------------------------------------------------------------

else:

    inner_fit_indices, inner_remaining_indices = train_test_split(
        train_indices,
        test_size=0.30,
        random_state=RANDOM_STATE,
        shuffle=True
    )

    inner_calibration_indices, inner_validation_indices = train_test_split(
        inner_remaining_indices,
        test_size=0.50,
        random_state=RANDOM_STATE,
        shuffle=True
    )

    np.savez(
        INNER_SPLIT_PATH,
        inner_fit_indices=inner_fit_indices,
        inner_calibration_indices=inner_calibration_indices,
        inner_validation_indices=inner_validation_indices,
        random_state=np.array([RANDOM_STATE])
    )

    print("已创建并保存新的Train内部划分：")
    print(INNER_SPLIT_PATH)


# ------------------------------------------------------------
# 3. 检查样本数量和是否重叠
# ------------------------------------------------------------

inner_fit_set = set(
    inner_fit_indices.tolist()
)

inner_calibration_set = set(
    inner_calibration_indices.tolist()
)

inner_validation_set = set(
    inner_validation_indices.tolist()
)

assert len(inner_fit_indices) == 1470
assert len(inner_calibration_indices) == 315
assert len(inner_validation_indices) == 315

assert inner_fit_set.isdisjoint(
    inner_calibration_set
)

assert inner_fit_set.isdisjoint(
    inner_validation_set
)

assert inner_calibration_set.isdisjoint(
    inner_validation_set
)

# 三个内部集合必须恰好组成外层Train
assert (
    inner_fit_set
    | inner_calibration_set
    | inner_validation_set
) == train_set

# 外层Calibration和Test不得进入内部调参数据
assert (
    inner_fit_set
    | inner_calibration_set
    | inner_validation_set
).isdisjoint(calibration_set)

assert (
    inner_fit_set
    | inner_calibration_set
    | inner_validation_set
).isdisjoint(test_set)


# ------------------------------------------------------------
# 4. 构造内部调参数据
# ------------------------------------------------------------

X_aspatial_inner_fit = X_aspatial.iloc[
    inner_fit_indices
].copy()

X_aspatial_inner_calibration = X_aspatial.iloc[
    inner_calibration_indices
].copy()

X_aspatial_inner_validation = X_aspatial.iloc[
    inner_validation_indices
].copy()


X_spatial_inner_fit = X_spatial.iloc[
    inner_fit_indices
].copy()

X_spatial_inner_calibration = X_spatial.iloc[
    inner_calibration_indices
].copy()

X_spatial_inner_validation = X_spatial.iloc[
    inner_validation_indices
].copy()


y_inner_fit = y.iloc[
    inner_fit_indices
].copy()

y_inner_calibration = y.iloc[
    inner_calibration_indices
].copy()

y_inner_validation = y.iloc[
    inner_validation_indices
].copy()


loc_inner_fit = loc.iloc[
    inner_fit_indices
].copy()

loc_inner_calibration = loc.iloc[
    inner_calibration_indices
].copy()

loc_inner_validation = loc.iloc[
    inner_validation_indices
].copy()


# ------------------------------------------------------------
# 5. 汇总内部划分
# ------------------------------------------------------------

inner_split_summary = pd.DataFrame({
    "Inner dataset": [
        "Inner Fit",
        "Inner Calibration",
        "Inner Validation"
    ],
    "Samples": [
        len(inner_fit_indices),
        len(inner_calibration_indices),
        len(inner_validation_indices)
    ],
    "Proportion of outer Train": [
        len(inner_fit_indices) / len(train_indices),
        len(inner_calibration_indices) / len(train_indices),
        len(inner_validation_indices) / len(train_indices)
    ],
    "Mean price": [
        y_inner_fit.mean(),
        y_inner_calibration.mean(),
        y_inner_validation.mean()
    ],
    "Median price": [
        y_inner_fit.median(),
        y_inner_calibration.median(),
        y_inner_validation.median()
    ]
})

display(
    inner_split_summary.round(4)
)

print("三个内部集合互不重叠：是")
print("内部样本总数：", (
    len(inner_fit_indices)
    + len(inner_calibration_indices)
    + len(inner_validation_indices)
))
print("外层Calibration未参与内部调参：是")
print("外层Test未参与内部调参：是")
print("内部划分保存位置：", INNER_SPLIT_PATH)

print(
    "\n第3章完成："
    "Train内部调参划分已经建立并保存。"
)

已读取以前保存的Train内部划分：
E:\GeoSIMCP_Reproduce\geoconformal-main\GCCP_Seattle\outputs\seattle_inner_split_seed42.npz


,Inner dataset,Samples,Proportion of outer Train,Mean price,Median price
0,Inner Fit,1470,0.70,53.0697,43.65
1,Inner Calibration,315,0.15,52.1961,42.50
2,Inner Validation,315,0.15,54.2139,42.50


三个内部集合互不重叠：是
内部样本总数： 2100
外层Calibration未参与内部调参：是
外层Test未参与内部调参：是
内部划分保存位置： E:\GeoSIMCP_Reproduce\geoconformal-main\GCCP_Seattle\outputs\seattle_inner_split_seed42.npz

第3章完成：Train内部调参划分已经建立并保存。


## 4. 统一XGBoost点预测模型

CP、GeoCP、GeoSIMCP和后续GCCP不是不同的点预测模型，而是在同一个点预测模型基础上采用不同的共形校准方式。

为保证公平比较，所有方法统一使用作者住房实验中的XGBoost回归模型。模型输入包括8个住房属性以及UTM_X、UTM_Y两个空间坐标，共10个特征。

本节训练两套参数相同的模型：

- **Inner Model**：使用Inner Fit训练，用于Train内部的GeoCP和GeoSIMCP超参数选择；
- **Final Model**：使用完整外层Train训练，用于参数确定后的正式Calibration和Test。

外层Test在本节中不参与模型评价。

In [10]:
# ============================================================
# 第4章：训练统一的XGBoost点预测模型
# ============================================================

# ------------------------------------------------------------
# 1. 保持作者住房实验中的主要模型参数
# ------------------------------------------------------------

XGBOOST_PARAMS = {
    "n_estimators": 500,
    "max_depth": 3,
    "min_child_weight": 1.0,
    "colsample_bytree": 1.0,
    "objective": "reg:squarederror",
    "random_state": RANDOM_STATE,
    "n_jobs": -1
}


# ------------------------------------------------------------
# 2. 训练内部临时模型
# 仅用于Train内部选择GeoCP和GeoSIMCP参数
# ------------------------------------------------------------

inner_model = xgb.XGBRegressor(
    **XGBOOST_PARAMS
)

inner_model.fit(
    X_spatial_inner_fit,
    y_inner_fit
)


# ------------------------------------------------------------
# 3. 在Inner Validation检查点预测能力
# Inner Calibration暂时不用于点预测评价
# ------------------------------------------------------------

y_pred_inner_validation = inner_model.predict(
    X_spatial_inner_validation
)

inner_validation_mae = mean_absolute_error(
    y_inner_validation,
    y_pred_inner_validation
)

inner_validation_rmse = np.sqrt(
    mean_squared_error(
        y_inner_validation,
        y_pred_inner_validation
    )
)

inner_validation_r2 = r2_score(
    y_inner_validation,
    y_pred_inner_validation
)


inner_model_performance = pd.DataFrame({
    "Metric": [
        "MAE",
        "RMSE",
        "R²"
    ],
    "Inner Validation result": [
        inner_validation_mae,
        inner_validation_rmse,
        inner_validation_r2
    ]
})

print("Inner Model点预测结果：")
display(
    inner_model_performance.round(4)
)


# ------------------------------------------------------------
# 4. 使用完整外层Train训练最终模型
# 目前不查看外层Test表现
# ------------------------------------------------------------

final_model = xgb.XGBRegressor(
    **XGBOOST_PARAMS
)

final_model.fit(
    X_spatial_train,
    y_train
)


# ------------------------------------------------------------
# 5. 基础检查
# ------------------------------------------------------------

assert inner_model.n_features_in_ == 10
assert final_model.n_features_in_ == 10

assert len(y_pred_inner_validation) == len(
    y_inner_validation
)

assert np.isfinite(
    y_pred_inner_validation
).all()


print("第4章完成。")
print(
    "Inner Model训练样本数：",
    len(y_inner_fit)
)
print(
    "Inner Validation样本数：",
    len(y_inner_validation)
)
print(
    "Final Model训练样本数：",
    len(y_train)
)
print(
    "模型输入特征数：",
    final_model.n_features_in_
)
print(
    "外层Calibration尚未正式校准：",
    len(y_calibration),
    "条"
)
print(
    "外层Test尚未评价：",
    len(y_test),
    "条"
)

Inner Model点预测结果：


,Metric,Inner Validation result
0,MAE,9.7474
1,RMSE,17.5917
2,R²,0.7941


第4章完成。
Inner Model训练样本数： 1470
Inner Validation样本数： 315
Final Model训练样本数： 2100
模型输入特征数： 10
外层Calibration尚未正式校准： 450 条
外层Test尚未评价： 450 条


### 4.1 Inner Model点预测结果说明

Inner Model在Inner Validation上的点预测结果为：

- MAE = 9.7474 万美元；
- RMSE = 17.5917 万美元；
- R² = 0.7941。

Inner Model仅使用1470条Inner Fit样本训练，因此其表现低于使用完整外层Train训练的模型属于正常现象。

RMSE明显高于MAE，说明Inner Validation中可能存在少量预测误差较大的住房样本。该模型只用于Train内部的共形参数选择，不用于最终测试集评价。

## 5. 传统Conformal Prediction内部基线

传统Split Conformal Prediction不考虑测试住房的地理位置和住房属性差异。

首先使用Inner Model预测Inner Calibration样本，并计算绝对残差：

\[
s_i = |y_i-\hat{y}_i|
\]

然后取全部校准残差的90%分位数作为统一区间半宽：

\[
q_{0.9}=\operatorname{Quantile}_{0.9}(s_1,\ldots,s_n)
\]

对于每个Inner Validation样本，其预测区间为：

\[
[\hat{y}-q_{0.9},\ \hat{y}+q_{0.9}]
\]

本节沿用作者公开代码中的普通分位数计算方式，以保证传统CP与后续GeoCP、GeoSIMCP的实现口径一致。

In [11]:
# ============================================================
# 第5章：传统CP的Train内部基线
# ============================================================

# ------------------------------------------------------------
# 1. 定义区间评分函数
# ------------------------------------------------------------

def interval_score(
    y_true,
    lower,
    upper,
    alpha=0.1
):
    """
    计算每个样本的Interval Score。

    分数越低越好：
    - 区间越窄，基础得分越低；
    - 真实值落在区间外时，给予额外惩罚。
    """
    y_true = np.asarray(y_true)
    lower = np.asarray(lower)
    upper = np.asarray(upper)

    width = upper - lower

    below_penalty = np.maximum(
        lower - y_true,
        0
    )

    above_penalty = np.maximum(
        y_true - upper,
        0
    )

    return (
        width
        + (2 / alpha) * below_penalty
        + (2 / alpha) * above_penalty
    )


# ------------------------------------------------------------
# 2. 定义统一评价函数
# 后续CP、GeoCP、GeoSIMCP和GCCP均使用它
# ------------------------------------------------------------

def evaluate_prediction_intervals(
    method_name,
    y_true,
    lower,
    upper,
    alpha=0.1
):
    y_true = np.asarray(y_true)
    lower = np.asarray(lower)
    upper = np.asarray(upper)

    covered = (
        (y_true >= lower)
        & (y_true <= upper)
    )

    widths = upper - lower

    return {
        "Method": method_name,
        "Coverage": float(np.mean(covered)),
        "Mean full width": float(np.mean(widths)),
        "Median full width": float(np.median(widths)),
        "Interval score": float(
            np.mean(
                interval_score(
                    y_true=y_true,
                    lower=lower,
                    upper=upper,
                    alpha=alpha
                )
            )
        )
    }


# ------------------------------------------------------------
# 3. 使用Inner Calibration计算传统CP校准残差
# ------------------------------------------------------------

y_pred_inner_calibration = inner_model.predict(
    X_spatial_inner_calibration
)

inner_calibration_scores = np.abs(
    np.asarray(y_inner_calibration)
    - y_pred_inner_calibration
)


# ------------------------------------------------------------
# 4. 沿用作者代码的90%普通分位数
# ------------------------------------------------------------

inner_cp_quantile = float(
    np.quantile(
        inner_calibration_scores,
        1 - ALPHA
    )
)


# ------------------------------------------------------------
# 5. 构造Inner Validation预测区间
# ------------------------------------------------------------

inner_cp_lower = (
    y_pred_inner_validation
    - inner_cp_quantile
)

inner_cp_upper = (
    y_pred_inner_validation
    + inner_cp_quantile
)


# ------------------------------------------------------------
# 6. 评价传统CP内部表现
# ------------------------------------------------------------

inner_cp_result = evaluate_prediction_intervals(
    method_name="CP",
    y_true=y_inner_validation,
    lower=inner_cp_lower,
    upper=inner_cp_upper,
    alpha=ALPHA
)

inner_cp_summary = pd.DataFrame([
    inner_cp_result
])

display(
    inner_cp_summary.round(4)
)


# ------------------------------------------------------------
# 7. 基础检查
# ------------------------------------------------------------

assert len(inner_calibration_scores) == 315
assert len(inner_cp_lower) == 315
assert len(inner_cp_upper) == 315

assert np.isfinite(
    inner_calibration_scores
).all()

assert np.isfinite(
    inner_cp_lower
).all()

assert np.isfinite(
    inner_cp_upper
).all()

assert np.all(
    inner_cp_upper >= inner_cp_lower
)


print("传统CP校准残差数量：", len(inner_calibration_scores))
print(
    "传统CP统一区间半宽：",
    round(inner_cp_quantile, 4),
    "万美元"
)
print(
    "传统CP完整区间宽度：",
    round(2 * inner_cp_quantile, 4),
    "万美元"
)
print("外层Calibration尚未使用：", len(y_calibration), "条")
print("外层Test尚未使用：", len(y_test), "条")
print("\n第5章完成：传统CP内部基线评价结束。")

,Method,Coverage,Mean full width,Median full width,Interval score
0,CP,0.8444,31.6447,31.6447,85.4405


传统CP校准残差数量： 315
传统CP统一区间半宽： 15.8223 万美元
传统CP完整区间宽度： 31.6447 万美元
外层Calibration尚未使用： 450 条
外层Test尚未使用： 450 条

第5章完成：传统CP内部基线评价结束。


## 6. GeoCP地理加权共形预测

传统CP对所有校准残差使用同一个全局分位数，而GeoCP根据测试住房与校准住房之间的地理距离，为校准残差赋予不同权重。

对于地理位置接近测试住房的校准样本，其残差权重更高；距离较远的校准样本权重更低。

本节首先使用一个固定bandwidth进行单次试运行，检查GeoCP代码接口和预测区间结果。确认运行正确后，再在Train内部搜索最优bandwidth。

外层Calibration和Test在本节中仍不参与参数选择。

In [12]:
# ============================================================
# 第6章第1步：GeoCP单一带宽试运行
# ============================================================

from GeoConformalSpatialRegression import (
    GeoConformalSpatialRegression
)


# ------------------------------------------------------------
# 1. 查看GeoCP类可使用的公开方法
# ------------------------------------------------------------

geocp_public_methods = [
    method_name
    for method_name in dir(
        GeoConformalSpatialRegression
    )
    if not method_name.startswith("_")
]

print("GeoCP公开方法：")
print(geocp_public_methods)


# ------------------------------------------------------------
# 2. 使用作者参数范围中的第一个带宽进行试运行
# ------------------------------------------------------------

smoke_bandwidth = 0.1

inner_geocp_smoke = GeoConformalSpatialRegression(
    predict_f=inner_model.predict,
    miscoverage_level=ALPHA,
    bandwidth=smoke_bandwidth,

    # Inner Calibration和Inner Validation的地理坐标
    coord_calib=loc_inner_calibration.to_numpy(),
    coord_test=loc_inner_validation.to_numpy(),

    # XGBoost预测使用10个特征
    X_calib=X_spatial_inner_calibration,
    y_calib=y_inner_calibration,

    X_test=X_spatial_inner_validation,
    y_test=y_inner_validation
)


# ------------------------------------------------------------
# 3. 运行GeoCP
# ------------------------------------------------------------

inner_geocp_smoke_result = (
    inner_geocp_smoke.analyze()
)


# ------------------------------------------------------------
# 4. 查看结果对象中包含哪些字段
# ------------------------------------------------------------

result_fields = [
    field_name
    for field_name in dir(
        inner_geocp_smoke_result
    )
    if not field_name.startswith("_")
]

print("\nGeoCP结果字段：")
print(result_fields)


# ------------------------------------------------------------
# 5. 提取预测区间
# ------------------------------------------------------------

geocp_smoke_lower = np.asarray(
    inner_geocp_smoke_result.lower_bound
)

geocp_smoke_upper = np.asarray(
    inner_geocp_smoke_result.upper_bound
)


# ------------------------------------------------------------
# 6. 使用统一函数评价
# ------------------------------------------------------------

geocp_smoke_metrics = (
    evaluate_prediction_intervals(
        method_name="GeoCP",
        y_true=y_inner_validation,
        lower=geocp_smoke_lower,
        upper=geocp_smoke_upper,
        alpha=ALPHA
    )
)

geocp_smoke_summary = pd.DataFrame([
    {
        "Method": "GeoCP",
        "Bandwidth": smoke_bandwidth,
        "Coverage": geocp_smoke_metrics["Coverage"],
        "Mean full width": geocp_smoke_metrics[
            "Mean full width"
        ],
        "Median full width": geocp_smoke_metrics[
            "Median full width"
        ],
        "Interval score": geocp_smoke_metrics[
            "Interval score"
        ]
    }
])

display(
    geocp_smoke_summary.round(4)
)


# ------------------------------------------------------------
# 7. 检查结果
# ------------------------------------------------------------

assert len(geocp_smoke_lower) == 315
assert len(geocp_smoke_upper) == 315

assert np.isfinite(
    geocp_smoke_lower
).all()

assert np.isfinite(
    geocp_smoke_upper
).all()

assert np.all(
    geocp_smoke_upper
    >= geocp_smoke_lower
)

print(
    "\nGeoCP单次试运行完成，"
    "bandwidth =",
    smoke_bandwidth
)

print(
    "外层Calibration尚未使用：",
    len(y_calibration),
    "条"
)

print(
    "外层Test尚未使用：",
    len(y_test),
    "条"
)

GeoCP公开方法：
['analyze', 'coverage_probability', 'predict_confidence_interval', 'predict_geoconformal_uncertainty']

GeoCP结果字段：
['alpha', 'betas', 'coords', 'coverage_probability', 'crs', 'geo_uncertainty', 'ks', 'lower_bound', 'pred', 'to_gpd', 'uncertainty', 'upper_bound']


,Method,Bandwidth,Coverage,Mean full width,Median full width,Interval score
0,GeoCP,0.1,0.8571,33.6271,35.1764,80.6478



GeoCP单次试运行完成，bandwidth = 0.1
外层Calibration尚未使用： 450 条
外层Test尚未使用： 450 条


### 6.1 GeoCP单次试运行结果

当 `bandwidth = 0.1` 时，GeoCP在Inner Validation上的结果为：

- Coverage = 0.8571；
- Mean Full Width = 33.6271万美元；
- Median Full Width = 35.1764万美元；
- Interval Score = 80.6478。

与传统CP相比，GeoCP的覆盖率有所提高，Interval Score有所降低，但当前覆盖率仍未达到目标值0.90。

本结果仅用于确认GeoCP代码接口和预测区间输出正确。接下来将在预设带宽范围内进行完整搜索，并根据统一规则选择GeoCP参数。

In [14]:
# ============================================================
# 第6章第2步：GeoCP完整带宽搜索
# ============================================================

# ------------------------------------------------------------
# 1. 设置带宽候选值
# ------------------------------------------------------------

geocp_bandwidth_values = np.linspace(
    0.1,
    5.0,
    20
)

geocp_validation_records = []


# ------------------------------------------------------------
# 2. 依次评价每个带宽
# ------------------------------------------------------------

for bandwidth in geocp_bandwidth_values:

    inner_geocp = GeoConformalSpatialRegression(
        predict_f=inner_model.predict,
        miscoverage_level=ALPHA,
        bandwidth=float(bandwidth),

        coord_calib=loc_inner_calibration.to_numpy(),
        coord_test=loc_inner_validation.to_numpy(),

        X_calib=X_spatial_inner_calibration,
        y_calib=y_inner_calibration,

        X_test=X_spatial_inner_validation,
        y_test=y_inner_validation
    )

    geocp_result = inner_geocp.analyze()

    geocp_lower = np.asarray(
        geocp_result.lower_bound
    )

    geocp_upper = np.asarray(
        geocp_result.upper_bound
    )

    metrics = evaluate_prediction_intervals(
        method_name="GeoCP",
        y_true=y_inner_validation,
        lower=geocp_lower,
        upper=geocp_upper,
        alpha=ALPHA
    )

    geocp_validation_records.append({
        "Method": "GeoCP",
        "Bandwidth": float(bandwidth),
        "Coverage": metrics["Coverage"],
        "Mean full width": metrics["Mean full width"],
        "Median full width": metrics["Median full width"],
        "Interval score": metrics["Interval score"]
    })


# ------------------------------------------------------------
# 3. 整理全部带宽结果
# ------------------------------------------------------------

geocp_validation_results = pd.DataFrame(
    geocp_validation_records
)

display(
    geocp_validation_results.round(4)
)


# ------------------------------------------------------------
# 4. 找出覆盖率达到目标的候选参数
# ------------------------------------------------------------

eligible_geocp_results = (
    geocp_validation_results[
        geocp_validation_results["Coverage"]
        >= TARGET_COVERAGE
    ]
    .copy()
)


# ------------------------------------------------------------
# 5. 按统一规则选择参数
# ------------------------------------------------------------

if len(eligible_geocp_results) > 0:

    eligible_geocp_results = (
        eligible_geocp_results
        .sort_values(
            by=[
                "Interval score",
                "Mean full width"
            ],
            ascending=True
        )
        .reset_index(drop=True)
    )

    best_geocp_row = (
        eligible_geocp_results.iloc[0]
    )

    best_geocp_bandwidth = float(
        best_geocp_row["Bandwidth"]
    )

    print("\n覆盖率达到0.90的GeoCP候选参数：")

    display(
        eligible_geocp_results
        .head(10)
        .round(4)
    )

    print(
        "\nGeoCP最佳Bandwidth：",
        round(best_geocp_bandwidth, 4)
    )

    print(
        "最佳Coverage：",
        round(
            float(best_geocp_row["Coverage"]),
            4
        )
    )

    print(
        "最佳Mean Full Width：",
        round(
            float(
                best_geocp_row["Mean full width"]
            ),
            4
        )
    )

    print(
        "最佳Interval Score：",
        round(
            float(
                best_geocp_row["Interval score"]
            ),
            4
        )
    )


else:

    best_geocp_bandwidth = None

    print(
        "\n当前0.1—5.0范围内，"
        "没有带宽在Inner Validation上达到0.90覆盖率。"
    )

    highest_coverage_results = (
        geocp_validation_results
        .sort_values(
            by=[
                "Coverage",
                "Interval score"
            ],
            ascending=[
                False,
                True
            ]
        )
        .reset_index(drop=True)
    )

    print("\n当前覆盖率最高的候选结果：")

    display(
        highest_coverage_results
        .head(10)
        .round(4)
    )


# ------------------------------------------------------------
# 6. 检查外层数据仍未被使用
# ------------------------------------------------------------

print(
    "\n外层Calibration尚未正式校准：",
    len(y_calibration),
    "条"
)

print(
    "外层Test尚未使用：",
    len(y_test),
    "条"
)

print(
    "\n第6章第2步完成："
    "GeoCP带宽搜索结束。"
)

,Method,Bandwidth,Coverage,Mean full width,Median full width,Interval score
0,GeoCP,0.1000,0.8571,33.6271,35.1764,80.6478
1,GeoCP,0.3579,0.8444,32.1888,32.3415,84.7984
2,GeoCP,0.6158,0.8444,31.8778,31.6747,85.3118
3,GeoCP,0.8737,0.8444,31.6885,31.6747,85.4166
4,GeoCP,1.1316,0.8444,31.6747,31.6747,85.4239
5,GeoCP,1.3895,0.8444,31.6747,31.6747,85.4239
6,GeoCP,1.6474,0.8444,31.6747,31.6747,85.4239
7,GeoCP,1.9053,0.8444,31.6747,31.6747,85.4239
8,GeoCP,2.1632,0.8444,31.6747,31.6747,85.4239
9,GeoCP,2.4211,0.8444,31.6747,31.6747,85.4239



当前0.1—5.0范围内，没有带宽在Inner Validation上达到0.90覆盖率。

当前覆盖率最高的候选结果：


,Method,Bandwidth,Coverage,Mean full width,Median full width,Interval score
0,GeoCP,0.1000,0.8571,33.6271,35.1764,80.6478
1,GeoCP,0.3579,0.8444,32.1888,32.3415,84.7984
2,GeoCP,0.6158,0.8444,31.8778,31.6747,85.3118
3,GeoCP,0.8737,0.8444,31.6885,31.6747,85.4166
4,GeoCP,1.1316,0.8444,31.6747,31.6747,85.4239
5,GeoCP,1.3895,0.8444,31.6747,31.6747,85.4239
6,GeoCP,1.6474,0.8444,31.6747,31.6747,85.4239
7,GeoCP,1.9053,0.8444,31.6747,31.6747,85.4239
8,GeoCP,2.1632,0.8444,31.6747,31.6747,85.4239
9,GeoCP,2.4211,0.8444,31.6747,31.6747,85.4239



外层Calibration尚未正式校准： 450 条
外层Test尚未使用： 450 条

第6章第2步完成：GeoCP带宽搜索结束。


In [15]:
# ============================================================
# 汇总GeoCP带宽搜索结果
# ============================================================

print("搜索的带宽数量：", len(geocp_validation_results))
print(
    "达到目标覆盖率的组合数：",
    len(eligible_geocp_results)
)

if best_geocp_bandwidth is not None:

    print("\nGeoCP最佳参数：")

    best_geocp_summary = pd.DataFrame([{
        "Bandwidth": best_geocp_bandwidth,
        "Coverage": float(best_geocp_row["Coverage"]),
        "Mean full width": float(
            best_geocp_row["Mean full width"]
        ),
        "Median full width": float(
            best_geocp_row["Median full width"]
        ),
        "Interval score": float(
            best_geocp_row["Interval score"]
        )
    }])

    display(best_geocp_summary.round(4))

    print("\n覆盖率达标的前10组参数：")

    display(
        eligible_geocp_results
        .head(10)
        .round(4)
    )

else:

    print(
        "\n当前0.1—5.0范围内，"
        "没有带宽达到0.90覆盖率。"
    )

    print("\n覆盖率最高的前10组参数：")

    display(
        geocp_validation_results
        .sort_values(
            by=["Coverage", "Interval score"],
            ascending=[False, True]
        )
        .head(10)
        .round(4)
    )

搜索的带宽数量： 20
达到目标覆盖率的组合数： 0

当前0.1—5.0范围内，没有带宽达到0.90覆盖率。

覆盖率最高的前10组参数：


,Method,Bandwidth,Coverage,Mean full width,Median full width,Interval score
0,GeoCP,0.1000,0.8571,33.6271,35.1764,80.6478
1,GeoCP,0.3579,0.8444,32.1888,32.3415,84.7984
2,GeoCP,0.6158,0.8444,31.8778,31.6747,85.3118
3,GeoCP,0.8737,0.8444,31.6885,31.6747,85.4166
4,GeoCP,1.1316,0.8444,31.6747,31.6747,85.4239
5,GeoCP,1.3895,0.8444,31.6747,31.6747,85.4239
6,GeoCP,1.6474,0.8444,31.6747,31.6747,85.4239
7,GeoCP,1.9053,0.8444,31.6747,31.6747,85.4239
8,GeoCP,2.1632,0.8444,31.6747,31.6747,85.4239
9,GeoCP,2.4211,0.8444,31.6747,31.6747,85.4239


### 6.2 GeoCP参数选择结果

在预设的20个bandwidth候选值中，没有参数在Inner Validation上达到目标覆盖率0.90。

因此，本实验启动统一备用选择规则：

1. 若存在覆盖率达到0.90的参数，则在达标参数中选择Interval Score最低者；
2. 若没有参数达到0.90，则选择覆盖率最高者；
3. 若覆盖率相同，则选择Interval Score最低者；
4. 若仍然相同，则选择平均完整区间宽度较小者。

根据该规则，GeoCP选择：

- bandwidth = 0.1000；
- Coverage = 0.8571；
- Mean Full Width = 33.6271万美元；
- Interval Score = 80.6478。

该参数仅根据外层Train内部的数据确定。外层Calibration和Test均未参与参数选择。

In [17]:
# ============================================================
# 第6章第3步：按照统一规则确定并保存GeoCP参数
# ============================================================

# ------------------------------------------------------------
# 1. 按照统一规则选择GeoCP参数
# ------------------------------------------------------------

if len(eligible_geocp_results) > 0:

    geocp_selection_status = "coverage_target_met"

    selected_geocp_row = (
        eligible_geocp_results
        .sort_values(
            by=[
                "Interval score",
                "Mean full width"
            ],
            ascending=[
                True,
                True
            ]
        )
        .iloc[0]
    )

else:

    geocp_selection_status = "fallback_highest_coverage"

    selected_geocp_row = (
        geocp_validation_results
        .sort_values(
            by=[
                "Coverage",
                "Interval score",
                "Mean full width"
            ],
            ascending=[
                False,
                True,
                True
            ]
        )
        .iloc[0]
    )


# ------------------------------------------------------------
# 2. 提取最终选择的带宽
# ------------------------------------------------------------

best_geocp_bandwidth = float(
    selected_geocp_row["Bandwidth"]
)


# ------------------------------------------------------------
# 3. 汇总最终选择结果
# ------------------------------------------------------------

geocp_selected_summary = pd.DataFrame([{
    "Method": "GeoCP",
    "Selection status": geocp_selection_status,
    "Bandwidth": best_geocp_bandwidth,
    "Coverage": float(
        selected_geocp_row["Coverage"]
    ),
    "Mean full width": float(
        selected_geocp_row["Mean full width"]
    ),
    "Median full width": float(
        selected_geocp_row["Median full width"]
    ),
    "Interval score": float(
        selected_geocp_row["Interval score"]
    )
}])

display(
    geocp_selected_summary.round(4)
)


# ------------------------------------------------------------
# 4. 保存GeoCP全部内部搜索结果
# ------------------------------------------------------------

GEOCP_SEARCH_PATH = (
    OUTPUT_DIR
    / "geocp_inner_search_seed42.csv"
)

geocp_validation_results.to_csv(
    GEOCP_SEARCH_PATH,
    index=False
)


# ------------------------------------------------------------
# 5. 输出保存信息
# ------------------------------------------------------------

print(
    "GeoCP最终选择的Bandwidth：",
    best_geocp_bandwidth
)

print(
    "参数选择状态：",
    geocp_selection_status
)

print(
    "GeoCP搜索结果保存位置：",
    GEOCP_SEARCH_PATH
)

print(
    "\n第6章完成：GeoCP参数已经确定并保存。"
)

,Method,Selection status,Bandwidth,Coverage,Mean full width,Median full width,Interval score
0,GeoCP,fallback_highest_coverage,0.1,0.8571,33.6271,35.1764,80.6478


GeoCP最终选择的Bandwidth： 0.1
参数选择状态： fallback_highest_coverage
GeoCP搜索结果保存位置： E:\GeoSIMCP_Reproduce\geoconformal-main\GCCP_Seattle\outputs\geocp_inner_search_seed42.csv

第6章完成：GeoCP参数已经确定并保存。


## 7. GeoSIMCP-EUC超参数选择

GeoSIMCP-EUC在GeoCP的基础上加入住房属性的欧氏距离。

其联合距离为：

\[
d_{\mathrm{joint}}
=
\sqrt{
\lambda d_{\mathrm{geo}}^2+
(1-\lambda)d_{\mathrm{EUC}}^2
}
\]

其中：

- `bandwidth`控制联合距离对应的局部权重衰减速度；
- `lambda = 1`时只使用地理距离，模型退化为GeoCP；
- `lambda = 0`时只使用住房属性距离；
- `0 < lambda < 1`时同时使用地理距离和属性距离。

本节首先设置 `bandwidth = 0.1`、`lambda = 0.5` 进行单次试运行，检查GeoSIMCP-EUC的代码接口和预测区间输出。

外层Calibration和Test仍不参与参数选择。

In [19]:
# ============================================================
# 第7章第1步：GeoSIMCP-EUC单次试运行
# ============================================================

from GeoSIMCP import (
    GeoSIMConformalSpatialRegression
    as GeoSIMCP_EUC
)


# ------------------------------------------------------------
# 1. 设置试运行参数
# ------------------------------------------------------------

euc_smoke_bandwidth = 0.1
euc_smoke_lambda = 0.5


# ------------------------------------------------------------
# 2. 构建GeoSIMCP-EUC
# ------------------------------------------------------------

inner_euc_smoke = GeoSIMCP_EUC(
    predict_f=inner_model.predict,
    miscoverage_level=ALPHA,
    bandwidth=euc_smoke_bandwidth,

    # 地理距离使用经纬度
    coord_calib=loc_inner_calibration.to_numpy(),
    coord_test=loc_inner_validation.to_numpy(),

    # XGBoost点预测使用10个输入特征
    X_calib=X_spatial_inner_calibration,
    y_calib=y_inner_calibration,

    X_test=X_spatial_inner_validation,
    y_test=y_inner_validation,

    # 欧氏属性距离使用8个非空间住房属性
    X_calib_weight=X_aspatial_inner_calibration,
    X_test_weight=X_aspatial_inner_validation,

    lambda_weight=euc_smoke_lambda,
    standardize_weights=True
)


# ------------------------------------------------------------
# 3. 运行GeoSIMCP-EUC
# ------------------------------------------------------------

inner_euc_smoke_result = inner_euc_smoke.analyze()


# ------------------------------------------------------------
# 4. 提取预测区间
# ------------------------------------------------------------

euc_smoke_lower = np.asarray(
    inner_euc_smoke_result.lower_bound
)

euc_smoke_upper = np.asarray(
    inner_euc_smoke_result.upper_bound
)


# ------------------------------------------------------------
# 5. 使用统一指标评价
# ------------------------------------------------------------

euc_smoke_metrics = evaluate_prediction_intervals(
    method_name="GeoSIMCP-EUC",
    y_true=y_inner_validation,
    lower=euc_smoke_lower,
    upper=euc_smoke_upper,
    alpha=ALPHA
)


euc_smoke_summary = pd.DataFrame([{
    "Method": "GeoSIMCP-EUC",
    "Bandwidth": euc_smoke_bandwidth,
    "Lambda": euc_smoke_lambda,
    "Coverage": euc_smoke_metrics["Coverage"],
    "Mean full width": euc_smoke_metrics[
        "Mean full width"
    ],
    "Median full width": euc_smoke_metrics[
        "Median full width"
    ],
    "Interval score": euc_smoke_metrics[
        "Interval score"
    ]
}])

display(
    euc_smoke_summary.round(4)
)


# ------------------------------------------------------------
# 6. 检查结果
# ------------------------------------------------------------

assert len(euc_smoke_lower) == len(
    y_inner_validation
)

assert len(euc_smoke_upper) == len(
    y_inner_validation
)

assert np.isfinite(euc_smoke_lower).all()
assert np.isfinite(euc_smoke_upper).all()

assert np.all(
    euc_smoke_upper >= euc_smoke_lower
)


print("GeoSIMCP-EUC单次试运行完成。")
print("试运行Bandwidth：", euc_smoke_bandwidth)
print("试运行Lambda：", euc_smoke_lambda)
print(
    "外层Calibration尚未使用：",
    len(y_calibration),
    "条"
)
print(
    "外层Test尚未使用：",
    len(y_test),
    "条"
)

,Method,Bandwidth,Lambda,Coverage,Mean full width,Median full width,Interval score
0,GeoSIMCP-EUC,0.1,0.5,0.4984,19.2985,11.6496,102.3648


GeoSIMCP-EUC单次试运行完成。
试运行Bandwidth： 0.1
试运行Lambda： 0.5
外层Calibration尚未使用： 450 条
外层Test尚未使用： 450 条


### 7.1 GeoSIMCP-EUC单次试运行结果

当 `bandwidth = 0.1`、`lambda = 0.5` 时，GeoSIMCP-EUC在Inner Validation上的结果为：

- Coverage = 0.4984；
- Mean Full Width = 19.2985万美元；
- Median Full Width = 11.6496万美元；
- Interval Score = 102.3648。

该参数组合产生了较窄的预测区间，但覆盖率严重不足，说明在加入8维住房属性欧氏距离后，`bandwidth = 0.1`对应的局部权重过于集中。

该结果仅为固定参数的接口试运行，不能作为GeoSIMCP-EUC的最终内部结果。正式参数需要通过bandwidth和lambda的联合搜索确定。

在开始联合搜索前，进一步检查当 `lambda = 1` 时，GeoSIMCP-EUC是否能够正确退化为GeoCP。

In [20]:
# ============================================================
# 第7章第2步：检查lambda=1时是否退化为GeoCP
# ============================================================

# ------------------------------------------------------------
# 1. 使用与GeoCP相同的bandwidth，并设置lambda=1
# ------------------------------------------------------------

euc_lambda1_check = GeoSIMCP_EUC(
    predict_f=inner_model.predict,
    miscoverage_level=ALPHA,
    bandwidth=best_geocp_bandwidth,

    coord_calib=loc_inner_calibration.to_numpy(),
    coord_test=loc_inner_validation.to_numpy(),

    X_calib=X_spatial_inner_calibration,
    y_calib=y_inner_calibration,

    X_test=X_spatial_inner_validation,
    y_test=y_inner_validation,

    X_calib_weight=X_aspatial_inner_calibration,
    X_test_weight=X_aspatial_inner_validation,

    lambda_weight=1.0,
    standardize_weights=True
)


# ------------------------------------------------------------
# 2. 运行GeoSIMCP-EUC
# ------------------------------------------------------------

euc_lambda1_result = euc_lambda1_check.analyze()

euc_lambda1_lower = np.asarray(
    euc_lambda1_result.lower_bound
)

euc_lambda1_upper = np.asarray(
    euc_lambda1_result.upper_bound
)


# ------------------------------------------------------------
# 3. 评价lambda=1时的结果
# ------------------------------------------------------------

euc_lambda1_metrics = evaluate_prediction_intervals(
    method_name="GeoSIMCP-EUC (lambda=1)",
    y_true=y_inner_validation,
    lower=euc_lambda1_lower,
    upper=euc_lambda1_upper,
    alpha=ALPHA
)


comparison_table = pd.DataFrame([
    {
        "Method": "GeoCP",
        "Bandwidth": best_geocp_bandwidth,
        "Lambda": np.nan,
        "Coverage": float(
            selected_geocp_row["Coverage"]
        ),
        "Mean full width": float(
            selected_geocp_row["Mean full width"]
        ),
        "Interval score": float(
            selected_geocp_row["Interval score"]
        )
    },
    {
        "Method": "GeoSIMCP-EUC",
        "Bandwidth": best_geocp_bandwidth,
        "Lambda": 1.0,
        "Coverage": euc_lambda1_metrics["Coverage"],
        "Mean full width": euc_lambda1_metrics[
            "Mean full width"
        ],
        "Interval score": euc_lambda1_metrics[
            "Interval score"
        ]
    }
])

display(comparison_table.round(6))


# ------------------------------------------------------------
# 4. 比较两种方法的区间端点
# ------------------------------------------------------------

max_lower_difference = float(
    np.max(
        np.abs(
            euc_lambda1_lower
            - geocp_smoke_lower
        )
    )
)

max_upper_difference = float(
    np.max(
        np.abs(
            euc_lambda1_upper
            - geocp_smoke_upper
        )
    )
)

print(
    "下界最大绝对差异：",
    max_lower_difference
)

print(
    "上界最大绝对差异：",
    max_upper_difference
)


# ------------------------------------------------------------
# 5. lambda=1时应当与GeoCP一致
# ------------------------------------------------------------

assert np.allclose(
    euc_lambda1_lower,
    geocp_smoke_lower,
    atol=1e-10
)

assert np.allclose(
    euc_lambda1_upper,
    geocp_smoke_upper,
    atol=1e-10
)

print(
    "\n一致性检查通过："
    "lambda=1时，GeoSIMCP-EUC正确退化为GeoCP。"
)

,Method,Bandwidth,Lambda,Coverage,Mean full width,Interval score
0,GeoCP,0.1,NaN,0.857143,33.627096,80.647767
1,GeoSIMCP-EUC,0.1,1.0,0.857143,33.627096,80.647767


下界最大绝对差异： 0.0
上界最大绝对差异： 0.0

一致性检查通过：lambda=1时，GeoSIMCP-EUC正确退化为GeoCP。


### 7.2 GeoSIMCP-EUC一致性检查

当 `lambda = 1`、`bandwidth = 0.1` 时，GeoSIMCP-EUC与GeoCP得到完全相同的预测区间：

- Coverage = 0.857143；
- Mean Full Width = 33.627096万美元；
- Interval Score = 80.647767；
- 上下界最大绝对差异均为0。

这说明GeoSIMCP-EUC实现满足理论退化关系：当特征距离权重为0时，GeoSIMCP-EUC退化为GeoCP。

因此，可以继续进行bandwidth和lambda的联合参数搜索。

In [21]:
# ============================================================
# 第7章第3步：GeoSIMCP-EUC完整参数搜索
# ============================================================

# ------------------------------------------------------------
# 1. 设置候选参数
# 与作者住房实验的搜索范围保持一致
# ------------------------------------------------------------

euc_bandwidth_values = np.linspace(
    0.1,
    5.0,
    20
)

euc_lambda_values = np.linspace(
    0.0,
    1.0,
    20
)

euc_validation_records = []

total_combinations = (
    len(euc_bandwidth_values)
    * len(euc_lambda_values)
)

completed_combinations = 0


# ------------------------------------------------------------
# 2. 搜索bandwidth和lambda
# ------------------------------------------------------------

for bandwidth_index, bandwidth in enumerate(
    euc_bandwidth_values,
    start=1
):

    for lambda_weight in euc_lambda_values:

        inner_euc_model = GeoSIMCP_EUC(
            predict_f=inner_model.predict,
            miscoverage_level=ALPHA,
            bandwidth=float(bandwidth),

            coord_calib=loc_inner_calibration.to_numpy(),
            coord_test=loc_inner_validation.to_numpy(),

            # 点预测模型使用10个输入特征
            X_calib=X_spatial_inner_calibration,
            y_calib=y_inner_calibration,

            X_test=X_spatial_inner_validation,
            y_test=y_inner_validation,

            # EUC距离使用8个非空间住房属性
            X_calib_weight=X_aspatial_inner_calibration,
            X_test_weight=X_aspatial_inner_validation,

            lambda_weight=float(lambda_weight),
            standardize_weights=True
        )

        euc_result = inner_euc_model.analyze()

        euc_lower = np.asarray(
            euc_result.lower_bound
        )

        euc_upper = np.asarray(
            euc_result.upper_bound
        )

        euc_metrics = evaluate_prediction_intervals(
            method_name="GeoSIMCP-EUC",
            y_true=y_inner_validation,
            lower=euc_lower,
            upper=euc_upper,
            alpha=ALPHA
        )

        euc_validation_records.append({
            "Method": "GeoSIMCP-EUC",
            "Bandwidth": float(bandwidth),
            "Lambda": float(lambda_weight),
            "Coverage": euc_metrics["Coverage"],
            "Mean full width": euc_metrics[
                "Mean full width"
            ],
            "Median full width": euc_metrics[
                "Median full width"
            ],
            "Interval score": euc_metrics[
                "Interval score"
            ]
        })

        completed_combinations += 1

    print(
        f"已完成 {completed_combinations}/"
        f"{total_combinations} 组参数，"
        f"当前bandwidth = {bandwidth:.4f}"
    )


# ------------------------------------------------------------
# 3. 整理全部搜索结果
# ------------------------------------------------------------

euc_validation_results = pd.DataFrame(
    euc_validation_records
)


# ------------------------------------------------------------
# 4. 保存完整搜索结果
# ------------------------------------------------------------

EUC_SEARCH_PATH = (
    OUTPUT_DIR
    / "geosimcp_euc_inner_search_seed42.csv"
)

euc_validation_results.to_csv(
    EUC_SEARCH_PATH,
    index=False
)


# ------------------------------------------------------------
# 5. 筛选达到目标覆盖率的候选参数
# ------------------------------------------------------------

eligible_euc_results = (
    euc_validation_results[
        euc_validation_results["Coverage"]
        >= TARGET_COVERAGE
    ]
    .copy()
)


# ------------------------------------------------------------
# 6. 按照统一规则选择参数
# ------------------------------------------------------------

if len(eligible_euc_results) > 0:

    euc_selection_status = "coverage_target_met"

    selected_euc_row = (
        eligible_euc_results
        .sort_values(
            by=[
                "Interval score",
                "Mean full width",
                "Bandwidth",
                "Lambda"
            ],
            ascending=[
                True,
                True,
                True,
                True
            ]
        )
        .iloc[0]
    )

else:

    euc_selection_status = (
        "fallback_highest_coverage"
    )

    selected_euc_row = (
        euc_validation_results
        .sort_values(
            by=[
                "Coverage",
                "Interval score",
                "Mean full width",
                "Bandwidth",
                "Lambda"
            ],
            ascending=[
                False,
                True,
                True,
                True,
                True
            ]
        )
        .iloc[0]
    )


# ------------------------------------------------------------
# 7. 提取最佳参数
# ------------------------------------------------------------

best_euc_bandwidth = float(
    selected_euc_row["Bandwidth"]
)

best_euc_lambda = float(
    selected_euc_row["Lambda"]
)


# ------------------------------------------------------------
# 8. 汇总最佳结果
# ------------------------------------------------------------

euc_selected_summary = pd.DataFrame([{
    "Method": "GeoSIMCP-EUC",
    "Selection status": euc_selection_status,
    "Bandwidth": best_euc_bandwidth,
    "Lambda": best_euc_lambda,
    "Coverage": float(
        selected_euc_row["Coverage"]
    ),
    "Mean full width": float(
        selected_euc_row["Mean full width"]
    ),
    "Median full width": float(
        selected_euc_row["Median full width"]
    ),
    "Interval score": float(
        selected_euc_row["Interval score"]
    )
}])

print("\nGeoSIMCP-EUC最终选择结果：")

display(
    euc_selected_summary.round(4)
)


# ------------------------------------------------------------
# 9. 显示最优候选参数
# ------------------------------------------------------------

if len(eligible_euc_results) > 0:

    print(
        "达到目标覆盖率的参数组合数：",
        len(eligible_euc_results)
    )

    print("\n达标参数中排名前10的组合：")

    euc_top_candidates = (
        eligible_euc_results
        .sort_values(
            by=[
                "Interval score",
                "Mean full width"
            ],
            ascending=True
        )
        .head(10)
    )

else:

    print(
        "达到目标覆盖率的参数组合数：0"
    )

    print("\n覆盖率最高的前10组参数：")

    euc_top_candidates = (
        euc_validation_results
        .sort_values(
            by=[
                "Coverage",
                "Interval score",
                "Mean full width"
            ],
            ascending=[
                False,
                True,
                True
            ]
        )
        .head(10)
    )

display(
    euc_top_candidates.round(4)
)


# ------------------------------------------------------------
# 10. 输出保存信息
# ------------------------------------------------------------

print(
    "\n最佳Bandwidth：",
    round(best_euc_bandwidth, 4)
)

print(
    "最佳Lambda：",
    round(best_euc_lambda, 4)
)

print(
    "参数选择状态：",
    euc_selection_status
)

print(
    "完整搜索结果保存位置：",
    EUC_SEARCH_PATH
)

print(
    "外层Calibration尚未使用：",
    len(y_calibration),
    "条"
)

print(
    "外层Test尚未使用：",
    len(y_test),
    "条"
)

print(
    "\n第7章第3步完成："
    "GeoSIMCP-EUC参数搜索结束。"
)

已完成 20/400 组参数，当前bandwidth = 0.1000
已完成 40/400 组参数，当前bandwidth = 0.3579
已完成 60/400 组参数，当前bandwidth = 0.6158
已完成 80/400 组参数，当前bandwidth = 0.8737
已完成 100/400 组参数，当前bandwidth = 1.1316
已完成 120/400 组参数，当前bandwidth = 1.3895
已完成 140/400 组参数，当前bandwidth = 1.6474
已完成 160/400 组参数，当前bandwidth = 1.9053
已完成 180/400 组参数，当前bandwidth = 2.1632
已完成 200/400 组参数，当前bandwidth = 2.4211
已完成 220/400 组参数，当前bandwidth = 2.6789
已完成 240/400 组参数，当前bandwidth = 2.9368
已完成 260/400 组参数，当前bandwidth = 3.1947
已完成 280/400 组参数，当前bandwidth = 3.4526
已完成 300/400 组参数，当前bandwidth = 3.7105
已完成 320/400 组参数，当前bandwidth = 3.9684
已完成 340/400 组参数，当前bandwidth = 4.2263
已完成 360/400 组参数，当前bandwidth = 4.4842
已完成 380/400 组参数，当前bandwidth = 4.7421
已完成 400/400 组参数，当前bandwidth = 5.0000

GeoSIMCP-EUC最终选择结果：


,Method,Selection status,Bandwidth,Lambda,Coverage,Mean full width,Median full width,Interval score
0,GeoSIMCP-EUC,fallback_highest_coverage,0.1,1.0,0.8571,33.6271,35.1764,80.6478


达到目标覆盖率的参数组合数：0

覆盖率最高的前10组参数：


,Method,Bandwidth,Lambda,Coverage,Mean full width,Median full width,Interval score
19,GeoSIMCP-EUC,0.1000,1.0000,0.8571,33.6271,35.1764,80.6478
90,GeoSIMCP-EUC,1.1316,0.5263,0.8476,31.0897,26.1543,63.6050
120,GeoSIMCP-EUC,1.6474,0.0000,0.8476,31.0669,26.1543,63.6386
74,GeoSIMCP-EUC,0.8737,0.7368,0.8444,30.9695,26.1706,63.8418
91,GeoSIMCP-EUC,1.1316,0.5789,0.8444,30.9542,26.1706,63.9632
107,GeoSIMCP-EUC,1.3895,0.3684,0.8444,30.9452,26.1706,63.9735
122,GeoSIMCP-EUC,1.6474,0.1053,0.8444,30.9391,26.1706,63.9743
124,GeoSIMCP-EUC,1.6474,0.2105,0.8444,30.8284,26.1706,64.0837
92,GeoSIMCP-EUC,1.1316,0.6316,0.8444,30.8751,26.3649,64.0982
38,GeoSIMCP-EUC,0.3579,0.9474,0.8444,31.5217,26.1543,64.1272



最佳Bandwidth： 0.1
最佳Lambda： 1.0
参数选择状态： fallback_highest_coverage
完整搜索结果保存位置： E:\GeoSIMCP_Reproduce\geoconformal-main\GCCP_Seattle\outputs\geosimcp_euc_inner_search_seed42.csv
外层Calibration尚未使用： 450 条
外层Test尚未使用： 450 条

第7章第3步完成：GeoSIMCP-EUC参数搜索结束。


### 7.3 GeoSIMCP-EUC内部参数选择结论

GeoSIMCP-EUC共搜索了400组bandwidth和lambda组合，但没有任何组合在Inner Validation上达到目标覆盖率0.90。

按照统一备用选择规则，最终选择：

- bandwidth = 0.1000；
- lambda = 1.0000；
- Coverage = 0.8571；
- Mean Full Width = 33.6271万美元；
- Median Full Width = 35.1764万美元；
- Interval Score = 80.6478。

由于 `lambda = 1`，属性欧氏距离的权重为0，因此本次选择的GeoSIMCP-EUC实际退化为GeoCP。

部分包含属性距离的参数组合获得了更低的Interval Score和更窄的区间。例如，`bandwidth = 1.1316`、`lambda = 0.5263`时，Interval Score为63.6050，但Coverage为0.8476，低于当前最高覆盖率0.8571，因此未被覆盖率优先规则选中。

该结果表明，在当前内部数据划分下，欧氏属性相似性能够提高区间效率，但同时降低了经验覆盖率。最终结论仍需使用固定参数和外层Calibration、Test进行评价。

## 8. GeoSIMCP-MND超参数选择

GeoSIMCP-MND同样结合地理距离和住房属性距离，但不使用欧氏距离，而是使用最大归一化属性差异。

对于测试住房 \(t\) 和校准住房 \(i\)，其MND属性距离为：

$$
d_{\mathrm{MND}}(i,t)
=
\max_{j}
\left[
\frac{
\left|x_{ij}-x_{tj}\right|
}{
\max(x_j)-\min(x_j)
}
\right]
$$

其中，\(j\) 表示住房属性。该距离选择所有属性中最大的归一化差异。

因此，只要某一个住房属性存在明显差异，两套住房的总体属性距离就会增大。

GeoSIMCP-MND的联合距离为：

$$
d_{\mathrm{joint}}(i,t)
=
\sqrt{
\lambda d_{\mathrm{geo}}(i,t)^2
+
(1-\lambda)d_{\mathrm{MND}}(i,t)^2
}
$$

其中：

- \(d_{\mathrm{geo}}\) 表示地理距离；
- \(d_{\mathrm{MND}}\) 表示最大归一化属性差异；
- \(\lambda\) 控制地理距离与属性距离的相对权重；
- 当 \(\lambda=1\) 时，GeoSIMCP-MND退化为GeoCP；
- 当 \(\lambda=0\) 时，只使用MND属性距离。

本节首先使用 `bandwidth = 0.1`、`lambda = 0.5` 进行单次试运行，确认GeoSIMCP-MND的接口和结果输出正确。

In [23]:
# ============================================================
# 第8章第1步：GeoSIMCP-MND单次试运行
# ============================================================

from GeoSIMCPzhao import (
    GeoSIMCPzhao as GeoSIMCP_MND
)


# ------------------------------------------------------------
# 1. 设置试运行参数
# ------------------------------------------------------------

mnd_smoke_bandwidth = 0.1
mnd_smoke_lambda = 0.5


# ------------------------------------------------------------
# 2. 构建GeoSIMCP-MND
# ------------------------------------------------------------

inner_mnd_smoke = GeoSIMCP_MND(
    predict_f=inner_model.predict,
    miscoverage_level=ALPHA,
    bandwidth=mnd_smoke_bandwidth,

    # 地理距离使用经纬度
    coord_calib=loc_inner_calibration.to_numpy(),
    coord_test=loc_inner_validation.to_numpy(),

    # XGBoost点预测使用10个特征
    X_calib=X_spatial_inner_calibration,
    y_calib=y_inner_calibration,

    X_test=X_spatial_inner_validation,
    y_test=y_inner_validation,

    # MND距离使用8个非空间住房属性
    X_calib_weight=X_aspatial_inner_calibration,
    X_test_weight=X_aspatial_inner_validation,

    lambda_weight=mnd_smoke_lambda,
    standardize_weights=True
)


# ------------------------------------------------------------
# 3. 运行GeoSIMCP-MND
# ------------------------------------------------------------

inner_mnd_smoke_result = inner_mnd_smoke.analyze()


# ------------------------------------------------------------
# 4. 提取预测区间
# ------------------------------------------------------------

mnd_smoke_lower = np.asarray(
    inner_mnd_smoke_result.lower_bound
)

mnd_smoke_upper = np.asarray(
    inner_mnd_smoke_result.upper_bound
)


# ------------------------------------------------------------
# 5. 使用统一指标评价
# ------------------------------------------------------------

mnd_smoke_metrics = evaluate_prediction_intervals(
    method_name="GeoSIMCP-MND",
    y_true=y_inner_validation,
    lower=mnd_smoke_lower,
    upper=mnd_smoke_upper,
    alpha=ALPHA
)


mnd_smoke_summary = pd.DataFrame([{
    "Method": "GeoSIMCP-MND",
    "Bandwidth": mnd_smoke_bandwidth,
    "Lambda": mnd_smoke_lambda,
    "Coverage": mnd_smoke_metrics["Coverage"],
    "Mean full width": mnd_smoke_metrics[
        "Mean full width"
    ],
    "Median full width": mnd_smoke_metrics[
        "Median full width"
    ],
    "Interval score": mnd_smoke_metrics[
        "Interval score"
    ]
}])

display(
    mnd_smoke_summary.round(4)
)


# ------------------------------------------------------------
# 6. 基础检查
# ------------------------------------------------------------

assert len(mnd_smoke_lower) == len(
    y_inner_validation
)

assert len(mnd_smoke_upper) == len(
    y_inner_validation
)

assert np.isfinite(mnd_smoke_lower).all()
assert np.isfinite(mnd_smoke_upper).all()

assert np.all(
    mnd_smoke_upper >= mnd_smoke_lower
)


print("GeoSIMCP-MND单次试运行完成。")
print("试运行Bandwidth：", mnd_smoke_bandwidth)
print("试运行Lambda：", mnd_smoke_lambda)
print(
    "外层Calibration尚未使用：",
    len(y_calibration),
    "条"
)
print(
    "外层Test尚未使用：",
    len(y_test),
    "条"
)

,Method,Bandwidth,Lambda,Coverage,Mean full width,Median full width,Interval score
0,GeoSIMCP-MND,0.1,0.5,0.8508,33.7031,26.1706,65.1982


GeoSIMCP-MND单次试运行完成。
试运行Bandwidth： 0.1
试运行Lambda： 0.5
外层Calibration尚未使用： 450 条
外层Test尚未使用： 450 条


### 8.1 GeoSIMCP-MND单次试运行结果

当 `bandwidth = 0.1`、`lambda = 0.5` 时，GeoSIMCP-MND在Inner Validation上的结果为：

- Coverage = 0.8508；
- Mean Full Width = 33.7031万美元；
- Median Full Width = 26.1706万美元；
- Interval Score = 65.1982。

与GeoCP相比，该参数组合的覆盖率略低，但Interval Score明显下降，说明MND属性距离可能改善区间效率。

不过，本结果只对应一组固定参数，不能作为GeoSIMCP-MND的最终结果。在进行完整参数搜索前，先检查当 `lambda = 1` 时，GeoSIMCP-MND是否能够正确退化为GeoCP。

In [24]:
# ============================================================
# 第8章第2步：检查lambda=1时MND是否退化为GeoCP
# ============================================================

# ------------------------------------------------------------
# 1. 设置lambda=1，只保留地理距离
# ------------------------------------------------------------

mnd_lambda1_check = GeoSIMCP_MND(
    predict_f=inner_model.predict,
    miscoverage_level=ALPHA,
    bandwidth=best_geocp_bandwidth,

    coord_calib=loc_inner_calibration.to_numpy(),
    coord_test=loc_inner_validation.to_numpy(),

    X_calib=X_spatial_inner_calibration,
    y_calib=y_inner_calibration,

    X_test=X_spatial_inner_validation,
    y_test=y_inner_validation,

    X_calib_weight=X_aspatial_inner_calibration,
    X_test_weight=X_aspatial_inner_validation,

    lambda_weight=1.0,
    standardize_weights=True
)


# ------------------------------------------------------------
# 2. 运行GeoSIMCP-MND
# ------------------------------------------------------------

mnd_lambda1_result = mnd_lambda1_check.analyze()

mnd_lambda1_lower = np.asarray(
    mnd_lambda1_result.lower_bound
)

mnd_lambda1_upper = np.asarray(
    mnd_lambda1_result.upper_bound
)


# ------------------------------------------------------------
# 3. 评价lambda=1时的结果
# ------------------------------------------------------------

mnd_lambda1_metrics = evaluate_prediction_intervals(
    method_name="GeoSIMCP-MND",
    y_true=y_inner_validation,
    lower=mnd_lambda1_lower,
    upper=mnd_lambda1_upper,
    alpha=ALPHA
)


mnd_lambda1_comparison = pd.DataFrame([
    {
        "Method": "GeoCP",
        "Bandwidth": best_geocp_bandwidth,
        "Lambda": np.nan,
        "Coverage": float(
            selected_geocp_row["Coverage"]
        ),
        "Mean full width": float(
            selected_geocp_row["Mean full width"]
        ),
        "Interval score": float(
            selected_geocp_row["Interval score"]
        )
    },
    {
        "Method": "GeoSIMCP-MND",
        "Bandwidth": best_geocp_bandwidth,
        "Lambda": 1.0,
        "Coverage": mnd_lambda1_metrics["Coverage"],
        "Mean full width": mnd_lambda1_metrics[
            "Mean full width"
        ],
        "Interval score": mnd_lambda1_metrics[
            "Interval score"
        ]
    }
])

display(
    mnd_lambda1_comparison.round(6)
)


# ------------------------------------------------------------
# 4. 比较两种方法的区间端点
# ------------------------------------------------------------

max_mnd_lower_difference = float(
    np.max(
        np.abs(
            mnd_lambda1_lower
            - geocp_smoke_lower
        )
    )
)

max_mnd_upper_difference = float(
    np.max(
        np.abs(
            mnd_lambda1_upper
            - geocp_smoke_upper
        )
    )
)

print(
    "下界最大绝对差异：",
    max_mnd_lower_difference
)

print(
    "上界最大绝对差异：",
    max_mnd_upper_difference
)


# ------------------------------------------------------------
# 5. lambda=1时必须与GeoCP一致
# ------------------------------------------------------------

assert np.allclose(
    mnd_lambda1_lower,
    geocp_smoke_lower,
    atol=1e-10
)

assert np.allclose(
    mnd_lambda1_upper,
    geocp_smoke_upper,
    atol=1e-10
)

print(
    "\n一致性检查通过："
    "lambda=1时，GeoSIMCP-MND正确退化为GeoCP。"
)

,Method,Bandwidth,Lambda,Coverage,Mean full width,Interval score
0,GeoCP,0.1,NaN,0.857143,33.627096,80.647767
1,GeoSIMCP-MND,0.1,1.0,0.857143,33.627096,80.647767


下界最大绝对差异： 0.0
上界最大绝对差异： 0.0

一致性检查通过：lambda=1时，GeoSIMCP-MND正确退化为GeoCP。


### 8.2 GeoSIMCP-MND一致性检查

当 `lambda = 1`、`bandwidth = 0.1` 时，GeoSIMCP-MND与GeoCP得到完全相同的预测区间：

- Coverage = 0.857143；
- Mean Full Width = 33.627096万美元；
- Interval Score = 80.647767；
- 上下界最大绝对差异均为0。

该结果验证了GeoSIMCP-MND的理论退化关系：当住房属性距离的权重为0时，GeoSIMCP-MND退化为GeoCP。

因此，接下来使用与GeoSIMCP-EUC相同的参数范围和选择规则，对GeoSIMCP-MND进行bandwidth与lambda联合搜索。

In [25]:
# ============================================================
# 第8章第3步：GeoSIMCP-MND完整参数搜索
# ============================================================

# ------------------------------------------------------------
# 1. 设置候选参数
# 与GeoSIMCP-EUC使用完全相同的搜索范围
# ------------------------------------------------------------

mnd_bandwidth_values = np.linspace(
    0.1,
    5.0,
    20
)

mnd_lambda_values = np.linspace(
    0.0,
    1.0,
    20
)

mnd_validation_records = []

total_mnd_combinations = (
    len(mnd_bandwidth_values)
    * len(mnd_lambda_values)
)

completed_mnd_combinations = 0


# ------------------------------------------------------------
# 2. 搜索bandwidth和lambda
# ------------------------------------------------------------

for bandwidth in mnd_bandwidth_values:

    for lambda_weight in mnd_lambda_values:

        inner_mnd_model = GeoSIMCP_MND(
            predict_f=inner_model.predict,
            miscoverage_level=ALPHA,
            bandwidth=float(bandwidth),

            # 地理距离使用经纬度
            coord_calib=loc_inner_calibration.to_numpy(),
            coord_test=loc_inner_validation.to_numpy(),

            # XGBoost点预测使用10个特征
            X_calib=X_spatial_inner_calibration,
            y_calib=y_inner_calibration,

            X_test=X_spatial_inner_validation,
            y_test=y_inner_validation,

            # MND距离使用8个非空间住房属性
            X_calib_weight=X_aspatial_inner_calibration,
            X_test_weight=X_aspatial_inner_validation,

            lambda_weight=float(lambda_weight),
            standardize_weights=True
        )

        mnd_result = inner_mnd_model.analyze()

        mnd_lower = np.asarray(
            mnd_result.lower_bound
        )

        mnd_upper = np.asarray(
            mnd_result.upper_bound
        )

        mnd_metrics = evaluate_prediction_intervals(
            method_name="GeoSIMCP-MND",
            y_true=y_inner_validation,
            lower=mnd_lower,
            upper=mnd_upper,
            alpha=ALPHA
        )

        mnd_validation_records.append({
            "Method": "GeoSIMCP-MND",
            "Bandwidth": float(bandwidth),
            "Lambda": float(lambda_weight),
            "Coverage": mnd_metrics["Coverage"],
            "Mean full width": mnd_metrics[
                "Mean full width"
            ],
            "Median full width": mnd_metrics[
                "Median full width"
            ],
            "Interval score": mnd_metrics[
                "Interval score"
            ]
        })

        completed_mnd_combinations += 1

    print(
        f"已完成 {completed_mnd_combinations}/"
        f"{total_mnd_combinations} 组参数，"
        f"当前bandwidth = {bandwidth:.4f}"
    )


# ------------------------------------------------------------
# 3. 整理全部搜索结果
# ------------------------------------------------------------

mnd_validation_results = pd.DataFrame(
    mnd_validation_records
)


# ------------------------------------------------------------
# 4. 保存完整搜索结果
# ------------------------------------------------------------

MND_SEARCH_PATH = (
    OUTPUT_DIR
    / "geosimcp_mnd_inner_search_seed42.csv"
)

mnd_validation_results.to_csv(
    MND_SEARCH_PATH,
    index=False
)


# ------------------------------------------------------------
# 5. 筛选达到目标覆盖率的候选参数
# ------------------------------------------------------------

eligible_mnd_results = (
    mnd_validation_results[
        mnd_validation_results["Coverage"]
        >= TARGET_COVERAGE
    ]
    .copy()
)


# ------------------------------------------------------------
# 6. 按照统一规则选择参数
# ------------------------------------------------------------

if len(eligible_mnd_results) > 0:

    mnd_selection_status = "coverage_target_met"

    selected_mnd_row = (
        eligible_mnd_results
        .sort_values(
            by=[
                "Interval score",
                "Mean full width",
                "Bandwidth",
                "Lambda"
            ],
            ascending=[
                True,
                True,
                True,
                True
            ]
        )
        .iloc[0]
    )

else:

    mnd_selection_status = (
        "fallback_highest_coverage"
    )

    selected_mnd_row = (
        mnd_validation_results
        .sort_values(
            by=[
                "Coverage",
                "Interval score",
                "Mean full width",
                "Bandwidth",
                "Lambda"
            ],
            ascending=[
                False,
                True,
                True,
                True,
                True
            ]
        )
        .iloc[0]
    )


# ------------------------------------------------------------
# 7. 提取最终参数
# ------------------------------------------------------------

best_mnd_bandwidth = float(
    selected_mnd_row["Bandwidth"]
)

best_mnd_lambda = float(
    selected_mnd_row["Lambda"]
)


# ------------------------------------------------------------
# 8. 汇总最终选择结果
# ------------------------------------------------------------

mnd_selected_summary = pd.DataFrame([{
    "Method": "GeoSIMCP-MND",
    "Selection status": mnd_selection_status,
    "Bandwidth": best_mnd_bandwidth,
    "Lambda": best_mnd_lambda,
    "Coverage": float(
        selected_mnd_row["Coverage"]
    ),
    "Mean full width": float(
        selected_mnd_row["Mean full width"]
    ),
    "Median full width": float(
        selected_mnd_row["Median full width"]
    ),
    "Interval score": float(
        selected_mnd_row["Interval score"]
    )
}])

print("\nGeoSIMCP-MND最终选择结果：")

display(
    mnd_selected_summary.round(4)
)


# ------------------------------------------------------------
# 9. 显示排名靠前的参数
# ------------------------------------------------------------

if len(eligible_mnd_results) > 0:

    print(
        "达到目标覆盖率的参数组合数：",
        len(eligible_mnd_results)
    )

    print("\n达标参数中排名前10的组合：")

    mnd_top_candidates = (
        eligible_mnd_results
        .sort_values(
            by=[
                "Interval score",
                "Mean full width"
            ],
            ascending=[
                True,
                True
            ]
        )
        .head(10)
    )

else:

    print(
        "达到目标覆盖率的参数组合数：0"
    )

    print("\n覆盖率最高的前10组参数：")

    mnd_top_candidates = (
        mnd_validation_results
        .sort_values(
            by=[
                "Coverage",
                "Interval score",
                "Mean full width"
            ],
            ascending=[
                False,
                True,
                True
            ]
        )
        .head(10)
    )

display(
    mnd_top_candidates.round(4)
)


# ------------------------------------------------------------
# 10. 输出保存信息
# ------------------------------------------------------------

print(
    "\n最佳Bandwidth：",
    round(best_mnd_bandwidth, 4)
)

print(
    "最佳Lambda：",
    round(best_mnd_lambda, 4)
)

print(
    "参数选择状态：",
    mnd_selection_status
)

print(
    "完整搜索结果保存位置：",
    MND_SEARCH_PATH
)

print(
    "外层Calibration尚未使用：",
    len(y_calibration),
    "条"
)

print(
    "外层Test尚未使用：",
    len(y_test),
    "条"
)

print(
    "\n第8章第3步完成："
    "GeoSIMCP-MND参数搜索结束。"
)

已完成 20/400 组参数，当前bandwidth = 0.1000
已完成 40/400 组参数，当前bandwidth = 0.3579
已完成 60/400 组参数，当前bandwidth = 0.6158
已完成 80/400 组参数，当前bandwidth = 0.8737
已完成 100/400 组参数，当前bandwidth = 1.1316
已完成 120/400 组参数，当前bandwidth = 1.3895
已完成 140/400 组参数，当前bandwidth = 1.6474
已完成 160/400 组参数，当前bandwidth = 1.9053
已完成 180/400 组参数，当前bandwidth = 2.1632
已完成 200/400 组参数，当前bandwidth = 2.4211
已完成 220/400 组参数，当前bandwidth = 2.6789
已完成 240/400 组参数，当前bandwidth = 2.9368
已完成 260/400 组参数，当前bandwidth = 3.1947
已完成 280/400 组参数，当前bandwidth = 3.4526
已完成 300/400 组参数，当前bandwidth = 3.7105
已完成 320/400 组参数，当前bandwidth = 3.9684
已完成 340/400 组参数，当前bandwidth = 4.2263
已完成 360/400 组参数，当前bandwidth = 4.4842
已完成 380/400 组参数，当前bandwidth = 4.7421
已完成 400/400 组参数，当前bandwidth = 5.0000

GeoSIMCP-MND最终选择结果：


,Method,Selection status,Bandwidth,Lambda,Coverage,Mean full width,Median full width,Interval score
0,GeoSIMCP-MND,fallback_highest_coverage,0.1,0.7895,0.8698,31.8677,27.021,62.8654


达到目标覆盖率的参数组合数：0

覆盖率最高的前10组参数：


,Method,Bandwidth,Lambda,Coverage,Mean full width,Median full width,Interval score
15,GeoSIMCP-MND,0.1,0.7895,0.8698,31.8677,27.0210,62.8654
13,GeoSIMCP-MND,0.1,0.6842,0.8667,33.2180,26.6942,63.7417
17,GeoSIMCP-MND,0.1,0.8947,0.8667,31.7882,27.9808,66.2135
14,GeoSIMCP-MND,0.1,0.7368,0.8635,32.2457,26.6942,63.5228
16,GeoSIMCP-MND,0.1,0.8421,0.8635,32.0115,27.3384,63.7576
12,GeoSIMCP-MND,0.1,0.6316,0.8635,33.4171,26.3649,63.9858
18,GeoSIMCP-MND,0.1,0.9474,0.8571,31.2521,31.6747,74.7788
19,GeoSIMCP-MND,0.1,1.0000,0.8571,33.6271,35.1764,80.6478
11,GeoSIMCP-MND,0.1,0.5789,0.8540,33.8053,26.3649,63.8312
10,GeoSIMCP-MND,0.1,0.5263,0.8508,33.9926,26.3649,64.5213



最佳Bandwidth： 0.1
最佳Lambda： 0.7895
参数选择状态： fallback_highest_coverage
完整搜索结果保存位置： E:\GeoSIMCP_Reproduce\geoconformal-main\GCCP_Seattle\outputs\geosimcp_mnd_inner_search_seed42.csv
外层Calibration尚未使用： 450 条
外层Test尚未使用： 450 条

第8章第3步完成：GeoSIMCP-MND参数搜索结束。


### 8.3 GeoSIMCP-MND内部参数选择结论

GeoSIMCP-MND共搜索了400组bandwidth和lambda组合，但没有参数在Inner Validation上达到目标覆盖率0.90。

按照统一备用选择规则，最终选择：

- bandwidth = 0.1000；
- lambda = 0.7895；
- Coverage = 0.8698；
- Mean Full Width = 31.8677万美元；
- Median Full Width = 27.0210万美元；
- Interval Score = 62.8654。

与GeoCP相比，GeoSIMCP-MND获得了更高的经验覆盖率、更窄的平均区间以及更低的Interval Score。

与GeoSIMCP-EUC最终选择的 `lambda = 1` 不同，GeoSIMCP-MND的最优 `lambda < 1`，说明MND住房属性距离在当前内部划分中提供了有效信息，模型没有退化为GeoCP。

不过，当前覆盖率仍未达到目标值0.90。以上结果只用于选择超参数，最终结论需要在参数完全固定后，使用外层Calibration正式校准，并在独立Test上进行一次评价。

## 9. 外层正式校准与独立测试

Train内部的超参数选择已经结束。GeoCP、GeoSIMCP-EUC和GeoSIMCP-MND的参数已经固定，后续不再根据外层Calibration或Test修改参数。

本章使用：

- 完整外层Train训练的Final Model；
- 外层Calibration计算正式校准残差；
- 外层Test进行一次最终评价。

外层Test从未参与模型训练或超参数选择，因此其结果可以用于公平比较CP、GeoCP、GeoSIMCP-EUC、GeoSIMCP-MND以及后续GCCP。

首先评价Final Model的点预测能力，并构造传统CP最终预测区间。

In [26]:
# ============================================================
# 第9章第1步：Final Model点预测和传统CP最终测试
# ============================================================

# ------------------------------------------------------------
# 1. 使用完整外层Train训练的Final Model进行预测
# ------------------------------------------------------------

y_pred_outer_calibration = final_model.predict(
    X_spatial_calibration
)

y_pred_outer_test = final_model.predict(
    X_spatial_test
)


# ------------------------------------------------------------
# 2. Final Model在独立Test上的点预测指标
# ------------------------------------------------------------

final_test_mae = mean_absolute_error(
    y_test,
    y_pred_outer_test
)

final_test_rmse = np.sqrt(
    mean_squared_error(
        y_test,
        y_pred_outer_test
    )
)

final_test_r2 = r2_score(
    y_test,
    y_pred_outer_test
)


final_point_summary = pd.DataFrame({
    "Metric": [
        "MAE",
        "RMSE",
        "R²"
    ],
    "Test result": [
        final_test_mae,
        final_test_rmse,
        final_test_r2
    ]
})

print("Final Model独立Test点预测结果：")

display(
    final_point_summary.round(4)
)


# ------------------------------------------------------------
# 3. 使用外层Calibration计算传统CP残差
# ------------------------------------------------------------

outer_calibration_scores = np.abs(
    np.asarray(y_calibration)
    - y_pred_outer_calibration
)


# ------------------------------------------------------------
# 4. 保持作者代码口径，使用普通90%分位数
# ------------------------------------------------------------

final_cp_quantile = float(
    np.quantile(
        outer_calibration_scores,
        1 - ALPHA
    )
)


# ------------------------------------------------------------
# 5. 构造外层Test上的传统CP区间
# ------------------------------------------------------------

final_cp_lower = (
    y_pred_outer_test
    - final_cp_quantile
)

final_cp_upper = (
    y_pred_outer_test
    + final_cp_quantile
)


# ------------------------------------------------------------
# 6. 评价传统CP最终结果
# ------------------------------------------------------------

final_cp_metrics = evaluate_prediction_intervals(
    method_name="CP",
    y_true=y_test,
    lower=final_cp_lower,
    upper=final_cp_upper,
    alpha=ALPHA
)


final_cp_summary = pd.DataFrame([{
    "Method": "CP",
    "Bandwidth": np.nan,
    "Lambda": np.nan,
    "Coverage": final_cp_metrics["Coverage"],
    "Mean full width": final_cp_metrics[
        "Mean full width"
    ],
    "Median full width": final_cp_metrics[
        "Median full width"
    ],
    "Interval score": final_cp_metrics[
        "Interval score"
    ]
}])

print("\n传统CP独立Test结果：")

display(
    final_cp_summary.round(4)
)


# ------------------------------------------------------------
# 7. 基础检查
# ------------------------------------------------------------

assert len(y_pred_outer_calibration) == 450
assert len(y_pred_outer_test) == 450
assert len(outer_calibration_scores) == 450

assert len(final_cp_lower) == 450
assert len(final_cp_upper) == 450

assert np.isfinite(
    y_pred_outer_test
).all()

assert np.isfinite(
    final_cp_lower
).all()

assert np.isfinite(
    final_cp_upper
).all()

assert np.all(
    final_cp_upper >= final_cp_lower
)


print(
    "\n外层Calibration残差数量：",
    len(outer_calibration_scores)
)

print(
    "传统CP最终区间半宽：",
    round(final_cp_quantile, 4),
    "万美元"
)

print(
    "传统CP最终完整区间宽度：",
    round(2 * final_cp_quantile, 4),
    "万美元"
)

print(
    "\n第9章第1步完成："
    "Final Model与传统CP独立测试结束。"
)

Final Model独立Test点预测结果：


,Metric,Test result
0,MAE,7.7265
1,RMSE,12.3241
2,R²,0.8476



传统CP独立Test结果：


,Method,Bandwidth,Lambda,Coverage,Mean full width,Median full width,Interval score
0,CP,NaN,NaN,0.9089,35.2126,35.2126,57.7319



外层Calibration残差数量： 450
传统CP最终区间半宽： 17.6063 万美元
传统CP最终完整区间宽度： 35.2126 万美元

第9章第1步完成：Final Model与传统CP独立测试结束。


### 9.1 Final Model与传统CP最终结果

Final Model在独立Test上的点预测结果为：

- MAE = 7.7265万美元；
- RMSE = 12.3241万美元；
- R² = 0.8476。

模型能够解释约84.76%的测试集房价变异，说明统一的XGBoost基础预测模型具有较好的预测能力。

传统CP使用外层Calibration中的450个绝对残差进行正式校准，在独立Test上的结果为：

- Coverage = 0.9089；
- Mean Full Width = 35.2126万美元；
- Median Full Width = 35.2126万美元；
- Interval Score = 57.7319。

传统CP达到目标覆盖率0.90。由于传统CP为全部测试样本使用相同的残差分位数，因此平均区间宽度与中位区间宽度完全相同。

后续GeoCP和GeoSIMCP将使用同一个Final Model、同一个外层Calibration和同一个Test，仅改变校准残差的局部加权方式。

In [28]:
# ============================================================
# 第9章第2步：GeoCP与GeoSIMCP独立Test最终评价
# ============================================================

# ------------------------------------------------------------
# 1. GeoCP最终评价
# ------------------------------------------------------------

final_geocp_model = GeoConformalSpatialRegression(
    predict_f=final_model.predict,
    miscoverage_level=ALPHA,
    bandwidth=best_geocp_bandwidth,

    coord_calib=loc_calibration.to_numpy(),
    coord_test=loc_test.to_numpy(),

    X_calib=X_spatial_calibration,
    y_calib=y_calibration,

    X_test=X_spatial_test,
    y_test=y_test
)

final_geocp_result = final_geocp_model.analyze()

final_geocp_lower = np.asarray(
    final_geocp_result.lower_bound
)

final_geocp_upper = np.asarray(
    final_geocp_result.upper_bound
)

final_geocp_metrics = evaluate_prediction_intervals(
    method_name="GeoCP",
    y_true=y_test,
    lower=final_geocp_lower,
    upper=final_geocp_upper,
    alpha=ALPHA
)


# ------------------------------------------------------------
# 2. GeoSIMCP-EUC最终评价
# ------------------------------------------------------------

final_euc_model = GeoSIMCP_EUC(
    predict_f=final_model.predict,
    miscoverage_level=ALPHA,
    bandwidth=best_euc_bandwidth,

    coord_calib=loc_calibration.to_numpy(),
    coord_test=loc_test.to_numpy(),

    X_calib=X_spatial_calibration,
    y_calib=y_calibration,

    X_test=X_spatial_test,
    y_test=y_test,

    X_calib_weight=X_aspatial_calibration,
    X_test_weight=X_aspatial_test,

    lambda_weight=best_euc_lambda,
    standardize_weights=True
)

final_euc_result = final_euc_model.analyze()

final_euc_lower = np.asarray(
    final_euc_result.lower_bound
)

final_euc_upper = np.asarray(
    final_euc_result.upper_bound
)

final_euc_metrics = evaluate_prediction_intervals(
    method_name="GeoSIMCP-EUC",
    y_true=y_test,
    lower=final_euc_lower,
    upper=final_euc_upper,
    alpha=ALPHA
)


# ------------------------------------------------------------
# 3. GeoSIMCP-MND最终评价
# ------------------------------------------------------------

final_mnd_model = GeoSIMCP_MND(
    predict_f=final_model.predict,
    miscoverage_level=ALPHA,
    bandwidth=best_mnd_bandwidth,

    coord_calib=loc_calibration.to_numpy(),
    coord_test=loc_test.to_numpy(),

    X_calib=X_spatial_calibration,
    y_calib=y_calibration,

    X_test=X_spatial_test,
    y_test=y_test,

    X_calib_weight=X_aspatial_calibration,
    X_test_weight=X_aspatial_test,

    lambda_weight=best_mnd_lambda,
    standardize_weights=True
)

final_mnd_result = final_mnd_model.analyze()

final_mnd_lower = np.asarray(
    final_mnd_result.lower_bound
)

final_mnd_upper = np.asarray(
    final_mnd_result.upper_bound
)

final_mnd_metrics = evaluate_prediction_intervals(
    method_name="GeoSIMCP-MND",
    y_true=y_test,
    lower=final_mnd_lower,
    upper=final_mnd_upper,
    alpha=ALPHA
)


# ------------------------------------------------------------
# 4. 汇总CP、GeoCP、EUC和MND最终结果
# ------------------------------------------------------------

final_baseline_results = pd.DataFrame([
    {
        "Method": "CP",
        "Bandwidth": np.nan,
        "Lambda": np.nan,
        "Coverage": final_cp_metrics["Coverage"],
        "Mean full width": final_cp_metrics[
            "Mean full width"
        ],
        "Median full width": final_cp_metrics[
            "Median full width"
        ],
        "Interval score": final_cp_metrics[
            "Interval score"
        ]
    },
    {
        "Method": "GeoCP",
        "Bandwidth": best_geocp_bandwidth,
        "Lambda": np.nan,
        "Coverage": final_geocp_metrics["Coverage"],
        "Mean full width": final_geocp_metrics[
            "Mean full width"
        ],
        "Median full width": final_geocp_metrics[
            "Median full width"
        ],
        "Interval score": final_geocp_metrics[
            "Interval score"
        ]
    },
    {
        "Method": "GeoSIMCP-EUC",
        "Bandwidth": best_euc_bandwidth,
        "Lambda": best_euc_lambda,
        "Coverage": final_euc_metrics["Coverage"],
        "Mean full width": final_euc_metrics[
            "Mean full width"
        ],
        "Median full width": final_euc_metrics[
            "Median full width"
        ],
        "Interval score": final_euc_metrics[
            "Interval score"
        ]
    },
    {
        "Method": "GeoSIMCP-MND",
        "Bandwidth": best_mnd_bandwidth,
        "Lambda": best_mnd_lambda,
        "Coverage": final_mnd_metrics["Coverage"],
        "Mean full width": final_mnd_metrics[
            "Mean full width"
        ],
        "Median full width": final_mnd_metrics[
            "Median full width"
        ],
        "Interval score": final_mnd_metrics[
            "Interval score"
        ]
    }
])

print("CP、GeoCP与GeoSIMCP独立Test最终结果：")

display(
    final_baseline_results.round(4)
)


# ------------------------------------------------------------
# 5. 一致性检查
# EUC的lambda=1，因此其结果必须与GeoCP相同
# ------------------------------------------------------------

if np.isclose(best_euc_lambda, 1.0):

    assert np.allclose(
        final_euc_lower,
        final_geocp_lower,
        atol=1e-10
    )

    assert np.allclose(
        final_euc_upper,
        final_geocp_upper,
        atol=1e-10
    )

    print(
        "\n一致性检查通过："
        "最终GeoSIMCP-EUC因lambda=1而退化为GeoCP。"
    )


# ------------------------------------------------------------
# 6. 基础结果检查
# ------------------------------------------------------------

for lower_array, upper_array in [
    (final_geocp_lower, final_geocp_upper),
    (final_euc_lower, final_euc_upper),
    (final_mnd_lower, final_mnd_upper)
]:

    assert len(lower_array) == len(y_test)
    assert len(upper_array) == len(y_test)

    assert np.isfinite(lower_array).all()
    assert np.isfinite(upper_array).all()

    assert np.all(
        upper_array >= lower_array
    )


# ------------------------------------------------------------
# 7. 保存最终baseline结果
# ------------------------------------------------------------

final_baseline_results.to_csv(
    BASELINE_RESULTS_PATH,
    index=False
)

print(
    "\n最终baseline结果保存位置：",
    BASELINE_RESULTS_PATH
)

print(
    "\n第9章第2步完成："
    "CP、GeoCP、GeoSIMCP-EUC和GeoSIMCP-MND"
    "独立测试结束。"
)

CP、GeoCP与GeoSIMCP独立Test最终结果：


,Method,Bandwidth,Lambda,Coverage,Mean full width,Median full width,Interval score
0,CP,NaN,NaN,0.9089,35.2126,35.2126,57.7319
1,GeoCP,0.1,NaN,0.9156,35.7454,37.1801,55.9272
2,GeoSIMCP-EUC,0.1,1.0000,0.9156,35.7454,37.1801,55.9272
3,GeoSIMCP-MND,0.1,0.7895,0.8933,32.3386,31.0050,49.0971



一致性检查通过：最终GeoSIMCP-EUC因lambda=1而退化为GeoCP。


NameError: name 'BASELINE_RESULTS_PATH' is not defined

In [29]:
# ============================================================
# 修复结果保存路径并保存最终baseline结果
# ============================================================

BASELINE_RESULTS_PATH = (
    OUTPUT_DIR
    / "baseline_results_seed42.csv"
)

# 确保输出目录存在
OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

# 再次显示已经计算好的结果
print("CP、GeoCP与GeoSIMCP独立Test最终结果：")

display(
    final_baseline_results.round(4)
)

# 保存结果
final_baseline_results.to_csv(
    BASELINE_RESULTS_PATH,
    index=False
)

print(
    "\n最终baseline结果已保存至：",
    BASELINE_RESULTS_PATH
)

print(
    "\n第9章第2步完成："
    "四种baseline独立测试结果已经保存。"
)

CP、GeoCP与GeoSIMCP独立Test最终结果：


,Method,Bandwidth,Lambda,Coverage,Mean full width,Median full width,Interval score
0,CP,NaN,NaN,0.9089,35.2126,35.2126,57.7319
1,GeoCP,0.1,NaN,0.9156,35.7454,37.1801,55.9272
2,GeoSIMCP-EUC,0.1,1.0000,0.9156,35.7454,37.1801,55.9272
3,GeoSIMCP-MND,0.1,0.7895,0.8933,32.3386,31.0050,49.0971



最终baseline结果已保存至： E:\GeoSIMCP_Reproduce\geoconformal-main\GCCP_Seattle\outputs\baseline_results_seed42.csv

第9章第2步完成：四种baseline独立测试结果已经保存。


### 9.2 四种Baseline最终结果分析

四种方法在独立Test上的最终结果如下：

- CP：Coverage = 0.9089，Mean Full Width = 35.2126，Interval Score = 57.7319；
- GeoCP：Coverage = 0.9156，Mean Full Width = 35.7454，Interval Score = 55.9272；
- GeoSIMCP-EUC：Coverage = 0.9156，Mean Full Width = 35.7454，Interval Score = 55.9272；
- GeoSIMCP-MND：Coverage = 0.8933，Mean Full Width = 32.3386，Interval Score = 49.0971。

CP和GeoCP均达到目标覆盖率0.90。GeoCP相对于CP略微增加了平均区间宽度，但获得了更高的覆盖率和更低的Interval Score，说明地理局部加权具有一定作用。

GeoSIMCP-EUC的最优lambda为1，因此其最终结果与GeoCP完全相同。这表明在当前数据划分和参数选择规则下，欧氏住房属性距离没有提供额外收益。

GeoSIMCP-MND获得了最窄的平均区间和最低的Interval Score，但Coverage为0.8933，略低于目标值0.90。因此，MND表现出较好的区间效率，但存在轻微欠覆盖，不能在覆盖率优先的评价标准下直接判定其优于GeoCP。

综上，在本次单次数据划分中：

- GeoCP是满足覆盖率要求的方法中综合表现最好的baseline；
- GeoSIMCP-MND是最有潜力的效率型baseline；
- GeoSIMCP-EUC退化为GeoCP。

In [30]:
# ============================================================
# 第9章第3步：保存每个Test样本的预测区间
# ============================================================

baseline_test_details = pd.DataFrame({
    # 原始数据索引与空间位置
    "original_index": test_indices,
    "longitude": loc_test["longitude"].to_numpy(),
    "latitude": loc_test["latitude"].to_numpy(),

    # 真实值与统一点预测
    "y_true": np.asarray(y_test),
    "y_pred": np.asarray(y_pred_outer_test),
    "absolute_error": np.abs(
        np.asarray(y_test) - np.asarray(y_pred_outer_test)
    ),

    # CP
    "cp_lower": final_cp_lower,
    "cp_upper": final_cp_upper,

    # GeoCP
    "geocp_lower": final_geocp_lower,
    "geocp_upper": final_geocp_upper,

    # GeoSIMCP-EUC
    "euc_lower": final_euc_lower,
    "euc_upper": final_euc_upper,

    # GeoSIMCP-MND
    "mnd_lower": final_mnd_lower,
    "mnd_upper": final_mnd_upper
})


# ------------------------------------------------------------
# 计算各方法的区间宽度和覆盖情况
# ------------------------------------------------------------

method_prefixes = [
    "cp",
    "geocp",
    "euc",
    "mnd"
]

for prefix in method_prefixes:

    lower_column = f"{prefix}_lower"
    upper_column = f"{prefix}_upper"

    baseline_test_details[
        f"{prefix}_width"
    ] = (
        baseline_test_details[upper_column]
        - baseline_test_details[lower_column]
    )

    baseline_test_details[
        f"{prefix}_covered"
    ] = (
        (
            baseline_test_details["y_true"]
            >= baseline_test_details[lower_column]
        )
        &
        (
            baseline_test_details["y_true"]
            <= baseline_test_details[upper_column]
        )
    )


# ------------------------------------------------------------
# 保存逐样本结果
# ------------------------------------------------------------

BASELINE_DETAIL_PATH = (
    OUTPUT_DIR
    / "baseline_test_intervals_seed42.csv"
)

baseline_test_details.to_csv(
    BASELINE_DETAIL_PATH,
    index=False
)


# ------------------------------------------------------------
# 检查保存结果
# ------------------------------------------------------------

display(
    baseline_test_details.head()
)

coverage_check = pd.DataFrame({
    "Method": [
        "CP",
        "GeoCP",
        "GeoSIMCP-EUC",
        "GeoSIMCP-MND"
    ],
    "Coverage from detail file": [
        baseline_test_details["cp_covered"].mean(),
        baseline_test_details["geocp_covered"].mean(),
        baseline_test_details["euc_covered"].mean(),
        baseline_test_details["mnd_covered"].mean()
    ],
    "Mean width from detail file": [
        baseline_test_details["cp_width"].mean(),
        baseline_test_details["geocp_width"].mean(),
        baseline_test_details["euc_width"].mean(),
        baseline_test_details["mnd_width"].mean()
    ]
})

print("逐样本结果复核：")

display(
    coverage_check.round(4)
)

assert len(baseline_test_details) == 450

print(
    "\n逐样本baseline结果保存位置：",
    BASELINE_DETAIL_PATH
)

print(
    "\n第9章完成：Baseline实验结果已经完整保存。"
)

,original_index,longitude,latitude,y_true,y_pred,absolute_error,cp_lower,cp_upper,geocp_lower,geocp_upper,...,mnd_lower,mnd_upper,cp_width,cp_covered,geocp_width,geocp_covered,euc_width,euc_covered,mnd_width,mnd_covered
0,1694,-122.170,47.4639,35.800,33.800846,1.999154,16.194559,51.407135,16.774261,50.827431,...,20.915207,46.686485,35.212578,True,34.053169,True,34.053169,True,25.771278,True
1,1450,-122.251,47.5078,27.200,38.052010,10.852010,20.445723,55.658295,19.461975,56.642044,...,25.433602,50.670417,35.212570,True,37.180069,True,37.180069,True,25.236814,True
2,1831,-122.390,47.6796,63.500,54.050652,9.449348,36.444366,71.656937,36.519096,71.582207,...,40.139603,67.961700,35.212570,True,35.063110,True,35.063110,True,27.822098,True
3,676,-122.380,47.2975,23.995,23.503937,0.491063,5.897650,41.110222,12.909026,34.098847,...,14.124739,32.883135,35.212570,True,21.189821,True,21.189821,True,18.758396,True
4,1298,-122.146,47.6186,76.800,72.613815,4.186185,55.007530,90.220100,47.991386,97.236244,...,47.920692,97.306938,35.212570,True,49.244858,True,49.244858,True,49.386246,True


逐样本结果复核：


,Method,Coverage from detail file,Mean width from detail file
0,CP,0.9089,35.2126
1,GeoCP,0.9156,35.7454
2,GeoSIMCP-EUC,0.9156,35.7454
3,GeoSIMCP-MND,0.8933,32.3386



逐样本baseline结果保存位置： E:\GeoSIMCP_Reproduce\geoconformal-main\GCCP_Seattle\outputs\baseline_test_intervals_seed42.csv

第9章完成：Baseline实验结果已经完整保存。
